# F0 - Preparação da base experimental CAMPI

Este notebook estabelece a base formal, documental e experimental utilizada por todas as fases posteriores da pesquisa de tradução de linguagem natural para a linguagem formal Nile. Sua responsabilidade é transformar o arquivo original do CAMPI em um conjunto de artefatos canônicos, validados, rastreáveis e reutilizáveis, sem executar o modelo professor nem o modelo aluno.

A única entrada externa é `extraction_campi.json`, disponibilizada no dataset do Kaggle. A partir dessa fonte, a F0 preserva o arquivo original, audita os 50 registros, documenta as alterações necessárias, consolida o subconjunto da Nile representado no corpus, implementa a cadeia de análise formal e prepara as divisões experimentais e os templates usados nas fases seguintes.

A cadeia principal da F0 é:

```text
extraction_campi.json
        ↓
auditoria da fonte
        ↓
padronização rastreável
        ↓
CAMPI canônico
        ↓
gramática Lark do subconjunto Nile
        ↓
parser → AST → verificação estrutural
        ↓
validação das 50 referências
        ↓
métricas, folds e templates
        ↓
f0_operacional.zip
```

A padronização é conservadora. O notebook não reinterpreta as intenções, não cria novas referências e não modifica silenciosamente o conteúdo. Toda alteração formal aplicada à Nile é registrada em `campi_changelog.csv`, com identificação do exemplo, valor anterior, valor posterior e justificativa operacional.

A gramática representa apenas o subconjunto da Nile observado e exigido pelo CAMPI. Ela não pretende descrever toda a linguagem Nile. Da mesma forma, os testes negativos são casos dirigidos de regressão: demonstram que classes específicas de erro são rejeitadas, mas não constituem prova matemática de correção, completude ou segurança para todas as expressões possíveis.

A F0 está organizada em dez blocos:

| Bloco | Etapa | Função metodológica | Resultado principal |
|---:|---|---|---|
| 1 | Configuração inicial | Fixar ambiente, diretórios, dependências, sementes e funções auxiliares | Base comum da execução |
| 2 | Carga e auditoria | Localizar, ler e verificar a fonte original | Arquivo original preservado e auditoria dos 50 registros |
| 3 | CAMPI canônico | Aplicar somente correções explícitas e rastreáveis | `campi_canonical.csv` e `campi_changelog.csv` |
| 4 | Gramática Nile | Formalizar o subconjunto observado no CAMPI | `nile_subset.lark` |
| 5 | Núcleo formal | Implementar parser, AST, renderização e verificação estrutural | `nile_core.py` |
| 6 | Validação | Testar o validador e conferir todas as referências | `validator_tests.jsonl` e `validation_references.csv` |
| 7 | Métricas | Definir o padrão comum de avaliação | `nile_metrics.py` |
| 8 | Folds | Construir cinco divisões reprodutíveis | `folds.csv` |
| 9 | Templates | Registrar os prompts das fases posteriores | `biblioteca_prompts.json` e `prompts/` |
| 10 | Empacotamento | Registrar proveniência, hashes e inventário | `manifest.json` e `f0_operacional.zip` |

Ao final, o pacote operacional contém os artefatos necessários para reproduzir a interpretação formal das referências, construir os experimentos e verificar a integridade das fases subsequentes.

## Bloco 1 - Configuração inicial

### Objetivo

Este bloco prepara o ambiente de execução da F0 e concentra as definições que deverão permanecer fixas durante toda a construção da base experimental. A centralização evita que caminhos, sementes, quantidades esperadas ou funções de escrita sejam redefinidos de maneira diferente nos blocos posteriores.

### Procedimentos executados

O bloco:

- importa as bibliotecas de sistema, manipulação tabular, serialização, hashing e visualização;
- verifica a disponibilidade da biblioteca Lark e realiza uma instalação controlada somente quando necessário;
- desativa a criação de bytecodes transitórios que não devem fazer parte do pacote final;
- configura a exibição do Pandas;
- identifica se a execução ocorre no Kaggle ou em outro ambiente compatível;
- define os diretórios de entrada, trabalho, saída operacional e pacote final;
- registra os caminhos de todos os artefatos produzidos pela F0;
- fixa o conjunto CAMPI, a quantidade de 50 exemplos, a semente global `42`, os cinco folds e os onze templates;
- cria funções auxiliares de leitura, escrita, serialização canônica, cálculo de SHA-256 e criação de ZIP;
- define o padrão visual e a numeração automática das tabelas.

### Reprodutibilidade

As funções de escrita usam UTF-8 e serialização estável. Os hashes são calculados sobre os arquivos efetivamente persistidos, permitindo que F1, F2 e F3 confirmem que estão consumindo exatamente os artefatos congelados nesta fase.

### Organização das saídas

Arquivos temporários permanecem separados dos artefatos operacionais. O diretório final só recebe arquivos explicitamente previstos. Caches, bytecodes e resultados transitórios não são considerados produtos da fase.

### Resultado esperado

Ao final, o ambiente estará configurado, os diretórios existirão e todas as funções comuns estarão disponíveis. O bloco não exibe tabelas, pois ainda não carrega dados nem produz resultados de auditoria.

In [1]:
# ----------------------------------------------------------
# 1.1 Importação das bibliotecas principais
# ----------------------------------------------------------

import ast
import csv
import hashlib
import importlib
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import py_compile
import random
import re
import shutil
import subprocess
import sys
import textwrap
import zipfile

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display


# ----------------------------------------------------------
# 1.2 Instalação controlada da dependência Lark
# ----------------------------------------------------------

def garantir_pacote(modulo: str, pacote_pip: str | None = None) -> None:
    """Instala um pacote somente quando o módulo ainda não está disponível."""
    pacote_pip = pacote_pip or modulo

    if importlib.util.find_spec(modulo) is not None:
        return

    resultado = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pacote_pip],
        capture_output=True,
        text=True,
        check=False,
    )

    importlib.invalidate_caches()

    if resultado.returncode != 0:
        raise RuntimeError(
            f"Não foi possível instalar o pacote '{pacote_pip}'. "
            f"Erro retornado pelo pip: {resultado.stderr.strip()}"
        )


garantir_pacote("lark", "lark")


# ----------------------------------------------------------
# 1.3 Importações que dependem da verificação anterior
# ----------------------------------------------------------

import lark
from sklearn.model_selection import StratifiedKFold


# ----------------------------------------------------------
# 1.4 Configuração de exibição do Pandas
# ----------------------------------------------------------

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 0)


# ----------------------------------------------------------
# 1.5 Definição do ambiente e dos diretórios principais
# ----------------------------------------------------------

KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")

BASE_DIR = Path(
    os.environ.get(
        "F0_OUTPUT_ROOT",
        str(KAGGLE_WORKING_DIR if KAGGLE_WORKING_DIR.exists() else Path.cwd())
    )
).resolve()

F0_DIR = BASE_DIR / "f0_operacional"
ZIP_F0_PATH = BASE_DIR / "f0_operacional.zip"

if F0_DIR.exists():
    shutil.rmtree(F0_DIR)

F0_DIR.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------
# 1.6 Definição dos caminhos dos artefatos
# ----------------------------------------------------------

CAMPI_ORIGINAL_PATH = F0_DIR / "extraction_campi.json"
CAMPI_CANONICAL_PATH = F0_DIR / "campi_canonical.csv"
CAMPI_CHANGELOG_PATH = F0_DIR / "campi_changelog.csv"
FOLDS_PATH = F0_DIR / "folds.csv"
GRAMMAR_PATH = F0_DIR / "nile_subset.lark"
NILE_CORE_PATH = F0_DIR / "nile_core.py"
NILE_METRICS_PATH = F0_DIR / "nile_metrics.py"
VALIDATOR_TESTS_PATH = F0_DIR / "validator_tests.jsonl"
VALIDATION_REFERENCES_PATH = F0_DIR / "validation_references.csv"
PROMPTS_DIR = F0_DIR / "prompts"
PROMPT_LIBRARY_PATH = F0_DIR / "biblioteca_prompts.json"
MANIFEST_PATH = F0_DIR / "manifest.json"

PROMPTS_DIR.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------
# 1.7 Parâmetros experimentais fixos
# ----------------------------------------------------------

FASE = "F0"
DATASET_ID = "CAMPI"
EXPECTED_EXAMPLES = 50
SEED = 42
N_FOLDS = 5
EXPECTED_TEST_PER_FOLD = 10
EXPECTED_TEMPLATES = 11

random.seed(SEED)


# ----------------------------------------------------------
# 1.8 Funções auxiliares de leitura, escrita e hash
# ----------------------------------------------------------

def salvar_json(objeto, caminho: Path) -> None:
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    with caminho.open("w", encoding="utf-8") as arquivo:
        json.dump(objeto, arquivo, ensure_ascii=False, indent=2)


def ler_json(caminho: Path):
    with Path(caminho).open("r", encoding="utf-8") as arquivo:
        return json.load(arquivo)


def salvar_jsonl(registros, caminho: Path) -> None:
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    with caminho.open("w", encoding="utf-8") as arquivo:
        for registro in registros:
            arquivo.write(json.dumps(registro, ensure_ascii=False) + "\n")


def calcular_sha256(caminho: Path) -> str:
    digest = hashlib.sha256()
    with Path(caminho).open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
            digest.update(bloco)
    return digest.hexdigest()


def versao_pacote(nome: str) -> str:
    try:
        return importlib_metadata.version(nome)
    except importlib_metadata.PackageNotFoundError:
        return "nao_instalado"


# ----------------------------------------------------------
# 1.9 Exibição padronizada e numeração das tabelas
# ----------------------------------------------------------

CONTADOR_TABELAS = 0

# Os nomes internos permanecem estáveis nos arquivos e no código.
# A tradução abaixo é aplicada somente à apresentação visual.
ROTULOS_COLUNAS = {
    "id": "ID",
    "dataset": "conjunto de dados",
    "arquivo_entrada": "arquivo de entrada",
    "registros": "registros",
    "textos_duplicados": "textos duplicados",
    "niles_duplicadas": "referências Nile duplicadas",
    "sha256": "SHA-256",
    "status": "status",
    "campos_faltantes": "campos ausentes",
    "university_ok": "universidade presente",
    "text_ok": "campo text presente",
    "nile_ok": "campo Nile presente",
    "parts_ok": "campo parts presente",
    "parts_reconstroem_texto": "parts reconstroem o texto",
    "n_parts": "segmentos em parts",
    "source_group": "grupo de origem",
    "source_index": "índice de origem",
    "university": "universidade",
    "nl": "NL",
    "nile_original": "Nile original",
    "nile_canonical": "Nile canônica",
    "nile_canonical_preliminary": "Nile canônica preliminar",
    "changed": "alterada",
    "change_types": "tipos de alteração",
    "change_description": "descrição da alteração",
    "review_status": "status da revisão",
    "primary_family": "família estrutural",
    "has_temporal": "possui restrição temporal",
    "has_route": "possui rota",
    "n_operations": "quantidade de operações",
    "n_items": "quantidade de itens",
    "complexity_score": "escore de complexidade",
    "media_operacoes": "média de operações",
    "media_itens": "média de itens",
    "media_complexidade": "média de complexidade",
    "category": "categoria",
    "expected_valid": "resultado esperado",
    "casos": "casos",
    "aprovados": "aprovados",
    "syntax_valid": "sintaxe válida",
    "structural_valid": "estrutura válida",
    "valid": "expressão válida",
    "roundtrip_ok": "ida e volta válida",
    "syntax_error_count": "erros sintáticos",
    "structural_error_count": "erros estruturais",
    "feedback": "diagnóstico",
    "case": "caso",
    "psr": "PSR",
    "em": "EM",
    "ed": "ED",
    "ned": "NED",
    "sla_s": "SLA-S",
    "sla_f": "SLA-F",
    "prediction_valid": "saída válida",
    "train": "treino",
    "test": "teste",
    "test_ids": "IDs de teste",
    "tp": "TP",
    "prompt_id": "ID do prompt",
    "fase": "fase",
    "modelo": "modelo",
    "estrategia": "estratégia",
    "nome": "nome",
    "arquivo": "arquivo",
    "objetivo": "objetivo",
    "placeholders": "placeholders",
    "exemplos": "exemplos",
    "referencias_sintaticamente_validas": "referências sintaticamente válidas",
    "referencias_estruturalmente_validas": "referências estruturalmente válidas",
    "psr_referencia": "PSR da referência",
    "folds": "folds",
    "testes_por_fold": "testes por fold",
    "prompts": "prompts",
    "arquivos_no_zip": "arquivos no ZIP",
    "zip": "ZIP",
}

ROTULOS_VALORES = {
    "correcao_formal_documentada": "correção formal documentada",
    "preservado": "preservada",
    "reordenacao_sintatica": "reordenação sintática",
    "correcao_temporal": "correção temporal",
    "remocao_espaco_externo": "remoção de espaço externo",
    "padronizacao_espaco_funcao": "padronização do espaço da função",
    "identica": "idêntica",
    "valor_alterado": "valor alterado",
    "rota_invertida": "rota invertida",
    "sintaxe_invalida": "sintaxe inválida",
    "ordem_acl_alterada": "ordem dos itens de ACL alterada",
    "ordem_middlebox_alterada": "ordem dos middleboxes alterada",
    "constraint_quota_alterada": "constraint de quota alterada",
    "espaco_interno_alterado": "espaço interno alterado",
    "positive_reference": "referência positiva",
    "acl": "ACL",
    "qos": "QoS",
    "mixed": "mista",
    "train": "treino",
    "test": "teste",
}


def traduzir_valores_para_exibicao(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepara uma cópia do DataFrame somente para apresentação.

    - booleanos são exibidos como sim/não;
    - ausências são exibidas como hífen;
    - categorias técnicas selecionadas recebem rótulos legíveis;
    - nomes internos e dados salvos permanecem inalterados.
    """

    df_exibicao = df.copy()

    def converter_valor(valor):
        if isinstance(valor, (bool, np.bool_)):
            return "sim" if valor else "não"

        if valor is None:
            return "-"

        try:
            if not isinstance(valor, str) and pd.isna(valor):
                return "-"
        except Exception:
            pass

        if isinstance(valor, str):
            valor_limpo = valor.strip()
            valor_lower = valor_limpo.lower()

            if valor_lower == "true":
                return "sim"
            if valor_lower == "false":
                return "não"
            if valor_lower in {
                "nan", "none", "null", "nat",
                "não se aplica", "nao se aplica", "n/a",
            }:
                return "-"

            return ROTULOS_VALORES.get(valor_limpo, valor)

        return valor

    for coluna in df_exibicao.columns:
        df_exibicao[coluna] = df_exibicao[coluna].apply(converter_valor)

    df_exibicao = df_exibicao.rename(
        columns={
            coluna: ROTULOS_COLUNAS.get(coluna, str(coluna).replace("_", " "))
            for coluna in df_exibicao.columns
        }
    )

    return df_exibicao


def exibir_tabela(
    df: pd.DataFrame,
    titulo: str = None,
    altura_px: int = None,
    largura_px: int = None,
    mostrar_indice: bool = False,
) -> None:
    """
    Exibe DataFrames com título numerado, cabeçalho fixo e rolagem.

    A rolagem vertical é usada somente quando altura_px é informado.
    """

    global CONTADOR_TABELAS

    if df is None:
        print("Tabela não disponível.")
        return

    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)

    if df.empty:
        print("Tabela vazia.")
        return

    CONTADOR_TABELAS += 1
    tabela = traduzir_valores_para_exibicao(df)

    titulo_final = f"Tabela {CONTADOR_TABELAS}"
    if titulo:
        titulo_final += f". {titulo}"

    html_tabela = tabela.to_html(
        escape=True,
        index=mostrar_indice,
        border=0,
        justify="left",
        classes="tabela_saida",
    )

    largura_css = f"{largura_px}px" if largura_px is not None else "100%"
    altura_css = (
        "overflow-y: visible;"
        if altura_px is None
        else f"max-height: {altura_px}px; overflow-y: auto;"
    )

    display(HTML(f"""
    <div style="
        font-weight: 600;
        font-size: 15px;
        margin-top: 8px;
        margin-bottom: 6px;
        color: #f1f1f1;
    ">
        {titulo_final}
    </div>

    <div class="container_tabela_saida" style="
        display: inline-block;
        width: {largura_css};
        max-width: 100%;
        overflow-x: auto;
        {altura_css}
        box-sizing: border-box;
        padding: 0;
        margin-top: 8px;
        margin-bottom: 12px;
        border: none;
        border-radius: 0;
    ">
        <style>
            .container_tabela_saida {{
                box-sizing: border-box !important;
            }}

            .container_tabela_saida table.tabela_saida {{
                border-collapse: collapse !important;
                table-layout: auto !important;
                width: auto !important;
                min-width: unset !important;
                max-width: none !important;
                font-family: Arial, sans-serif !important;
                font-size: 13px !important;
                background-color: #111 !important;
                color: #f1f1f1 !important;
                border: 1px solid #555 !important;
            }}

            .container_tabela_saida table.tabela_saida thead th {{
                position: sticky !important;
                top: 0 !important;
                z-index: 2 !important;
                background-color: #2b2b2b !important;
                color: #ffffff !important;
                font-weight: bold !important;
                padding: 7px !important;
                text-align: left !important;
                white-space: nowrap !important;
                border: 1px solid #777 !important;
                border-bottom: 2px solid #888 !important;
            }}

            .container_tabela_saida table.tabela_saida tbody td {{
                padding: 7px !important;
                vertical-align: top !important;
                text-align: left !important;
                white-space: nowrap !important;
                border: 1px solid #555 !important;
            }}

            .container_tabela_saida table.tabela_saida tbody tr:nth-child(even) td {{
                background-color: #1b1b1b !important;
            }}

            .container_tabela_saida table.tabela_saida tbody tr:nth-child(odd) td {{
                background-color: #111 !important;
            }}
        </style>
        {html_tabela}
    </div>
    """))


# ----------------------------------------------------------
# 1.10 Saída do bloco
# ----------------------------------------------------------

print("Bloco 1 concluído")
print(f"Fase: {FASE}")
print(f"Dataset esperado: {DATASET_ID}")
print(f"Diretório de saída: {F0_DIR}")
print(f"Semente dos folds: {SEED}")
print("Status: OK")

Bloco 1 concluído
Fase: F0
Dataset esperado: CAMPI
Diretório de saída: /kaggle/working/f0_operacional
Semente dos folds: 42
Status: OK


## Bloco 2 - Carga e auditoria do CAMPI

### Objetivo

Este bloco localiza e audita a única fonte externa da F0: `extraction_campi.json`. A intenção é confirmar a integridade mínima do corpus antes de qualquer transformação e preservar uma cópia imutável da fonte utilizada no experimento.

### Localização da entrada

O arquivo é procurado nos diretórios de entrada configurados, incluindo os datasets montados pelo Kaggle. A localização automática evita dependência de um nome específico do dataset, mas exige que o arquivo correto esteja presente e seja único entre os candidatos válidos.

### Verificações realizadas

A auditoria confere:

- se a raiz do JSON possui o formato esperado;
- se existem exatamente 50 registros;
- se cada registro contém os campos obrigatórios;
- se os identificadores e conteúdos podem ser lidos sem perda;
- se a entrada em linguagem natural e a expressão Nile são textos não vazios;
- se há registros duplicados de forma exata;
- se a sequência de exemplos pode ser associada aos IDs operacionais `campi_001` a `campi_050`.

Nenhuma correção é aplicada neste bloco. Inconsistências são registradas ou interrompem a execução, conforme sua gravidade.

### Preservação da fonte

O arquivo localizado é copiado para a área operacional com o nome `extraction_campi.json`. Essa cópia preservada funciona como referência primária da F0 e permite comparar a fonte original com o corpus canônico e o changelog.

### Tabelas produzidas

O bloco apresenta:

- a origem efetivamente localizada;
- o tamanho e o hash do arquivo;
- o resultado da auditoria de cada registro;
- a presença ou ausência de duplicações exatas.

### Resultado esperado

A execução só avança quando os 50 registros podem ser lidos e identificados. O resultado é uma fonte preservada e uma auditoria explícita, sem qualquer alteração do conteúdo original.

In [2]:
# ----------------------------------------------------------
# 2.1 Localização automática do extraction_campi.json
# ----------------------------------------------------------

def localizar_campi() -> Path:
    caminho_manual = os.environ.get("CAMPI_PATH")
    if caminho_manual:
        caminho = Path(caminho_manual).expanduser().resolve()
        if not caminho.exists():
            raise FileNotFoundError(f"CAMPI_PATH aponta para um arquivo inexistente: {caminho}")
        return caminho

    raizes = []
    if KAGGLE_INPUT_DIR.exists():
        raizes.append(KAGGLE_INPUT_DIR)
    raizes.append(Path.cwd())

    candidatos = []
    for raiz in raizes:
        candidatos.extend(raiz.rglob("extraction_campi.json"))

    candidatos = sorted({c.resolve() for c in candidatos if c.is_file()})

    if not candidatos:
        raise FileNotFoundError(
            "O arquivo extraction_campi.json não foi encontrado. "
            "Adicione o Dataset campi_dataset ao notebook do Kaggle."
        )

    preferidos = [
        caminho for caminho in candidatos
        if "campi" in str(caminho.parent).lower()
    ]

    if len(preferidos) == 1:
        return preferidos[0]
    if len(candidatos) == 1:
        return candidatos[0]

    raise RuntimeError(
        "Foram encontrados vários arquivos extraction_campi.json. "
        "Defina a variável de ambiente CAMPI_PATH com o caminho correto.\n"
        + "\n".join(str(c) for c in candidatos)
    )


CAMPI_INPUT_PATH = localizar_campi()


# ----------------------------------------------------------
# 2.2 Leitura e validação da estrutura raiz
# ----------------------------------------------------------

campi_raw = ler_json(CAMPI_INPUT_PATH)

if not isinstance(campi_raw, dict):
    raise TypeError("O arquivo CAMPI deve conter um objeto JSON na raiz.")

if "intents" not in campi_raw:
    raise KeyError("A chave obrigatória 'intents' não foi encontrada no CAMPI.")

intents = campi_raw["intents"]

if not isinstance(intents, list):
    raise TypeError("A chave 'intents' deve conter uma lista.")

if len(intents) != EXPECTED_EXAMPLES:
    raise ValueError(
        f"Quantidade inesperada de registros: {len(intents)}. "
        f"Esperado: {EXPECTED_EXAMPLES}."
    )


# ----------------------------------------------------------
# 2.3 Auditoria individual dos registros
# ----------------------------------------------------------

campos_obrigatorios = {"university", "text", "nile", "parts"}
registros_auditoria = []

for indice, registro in enumerate(intents, start=1):
    faltantes = sorted(campos_obrigatorios - set(registro))
    parts = registro.get("parts")
    parts_valid = isinstance(parts, list) and len(parts) > 0
    texto_parts = "".join(
        str(parte.get("text", ""))
        for parte in parts
    ) if parts_valid else ""

    registros_auditoria.append({
        "id": f"campi_{indice:03d}",
        "campos_faltantes": ", ".join(faltantes),
        "university_ok": isinstance(registro.get("university"), str) and bool(registro.get("university", "").strip()),
        "text_ok": isinstance(registro.get("text"), str) and bool(registro.get("text", "").strip()),
        "nile_ok": isinstance(registro.get("nile"), str) and bool(registro.get("nile", "").strip()),
        "parts_ok": parts_valid,
        "parts_reconstroem_texto": texto_parts == registro.get("text", ""),
        "n_parts": len(parts) if isinstance(parts, list) else 0,
    })

df_auditoria = pd.DataFrame(registros_auditoria)

colunas_booleanas_obrigatorias = [
    "university_ok",
    "text_ok",
    "nile_ok",
    "parts_ok",
]

if df_auditoria["campos_faltantes"].ne("").any():
    raise ValueError("Existem registros com campos obrigatórios ausentes.")

if not df_auditoria[colunas_booleanas_obrigatorias].all().all():
    problemas = df_auditoria.loc[
        ~df_auditoria[colunas_booleanas_obrigatorias].all(axis=1)
    ]
    raise ValueError(
        "A auditoria estrutural encontrou registros inválidos:\n"
        + problemas.to_string(index=False)
    )


# ----------------------------------------------------------
# 2.4 Verificação de duplicações exatas
# ----------------------------------------------------------

textos = [registro["text"] for registro in intents]
niles = [registro["nile"] for registro in intents]

textos_duplicados = len(textos) - len(set(textos))
niles_duplicadas = len(niles) - len(set(niles))


# ----------------------------------------------------------
# 2.5 Preservação do arquivo original
# ----------------------------------------------------------

shutil.copy2(CAMPI_INPUT_PATH, CAMPI_ORIGINAL_PATH)

if calcular_sha256(CAMPI_INPUT_PATH) != calcular_sha256(CAMPI_ORIGINAL_PATH):
    raise RuntimeError("A cópia do CAMPI original não preservou o hash SHA-256.")


# ----------------------------------------------------------
# 2.6 Tabela da fonte carregada
# ----------------------------------------------------------

resumo_fonte = pd.DataFrame([{
    "dataset": DATASET_ID,
    "arquivo_entrada": str(CAMPI_INPUT_PATH),
    "registros": len(intents),
    "textos_duplicados": textos_duplicados,
    "niles_duplicadas": niles_duplicadas,
    "sha256": calcular_sha256(CAMPI_INPUT_PATH),
    "status": "OK",
}])

exibir_tabela(
    resumo_fonte,
    "Fonte CAMPI carregada e preservada",
    altura_px=220,
)


# ----------------------------------------------------------
# 2.7 Tabela da auditoria dos registros
# ----------------------------------------------------------

exibir_tabela(
    df_auditoria,
    "Auditoria estrutural dos 50 registros do CAMPI",
    altura_px=500,
)


# ----------------------------------------------------------
# 2.8 Saída do bloco
# ----------------------------------------------------------

print("Bloco 2 concluído")
print(f"Registros esperados: {EXPECTED_EXAMPLES}")
print(f"Registros obtidos: {len(intents)}")
print(f"Arquivo original preservado em: {CAMPI_ORIGINAL_PATH}")
print("Status: OK")

conjunto de dados,arquivo de entrada,registros,textos duplicados,referências Nile duplicadas,SHA-256,status
CAMPI,/kaggle/input/datasets/thiagoarajoguedes/campi-dataset/extraction_campi.json,50,0,2,590161a042b50f4bb18177bae83a7a3da353025715a93780746d8905e9427ede,OK


ID,campos ausentes,universidade presente,campo text presente,campo Nile presente,campo parts presente,parts reconstroem o texto,segmentos em parts
campi_001,,sim,sim,sim,sim,sim,5
campi_002,,sim,sim,sim,sim,não,7
campi_003,,sim,sim,sim,sim,não,4
campi_004,,sim,sim,sim,sim,sim,7
campi_005,,sim,sim,sim,sim,sim,9
campi_006,,sim,sim,sim,sim,sim,7
campi_007,,sim,sim,sim,sim,sim,7
campi_008,,sim,sim,sim,sim,sim,7
campi_009,,sim,sim,sim,sim,sim,9
campi_010,,sim,sim,sim,sim,sim,9


Bloco 2 concluído
Registros esperados: 50
Registros obtidos: 50
Arquivo original preservado em: /kaggle/working/f0_operacional/extraction_campi.json
Status: OK


## Bloco 3 - Construção do CAMPI canônico

### Objetivo

Este bloco produz a versão operacional do CAMPI utilizada nas fases seguintes. O procedimento é denominado padronização porque atua somente sobre aspectos formais necessários à análise automática, sem alterar o significado das intenções ou criar novas referências.

### Política de padronização

São permitidas duas classes de intervenção:

1. padronização segura de espaços, que remove variações puramente tipográficas;
2. correções formais explícitas, aplicadas individualmente quando a referência original não é compatível com a sintaxe consolidada ou contém uma forma reconhecidamente inconsistente.

Cada correção explícita é documentada. Não existe substituição silenciosa, inferência automática de intenção ou reescrita livre da Nile.

### Construção da base

Para cada registro, o bloco:

- atribui um ID operacional estável;
- preserva a entrada em linguagem natural;
- mantém a Nile original para rastreabilidade;
- aplica as regras de padronização;
- registra a Nile operacional preliminar;
- calcula hashes e metadados necessários à auditoria;
- associa cada alteração ao respectivo motivo.

### Artefatos produzidos

São criados inicialmente:

- `campi_canonical.csv`, com a base operacional;
- `campi_changelog.csv`, com o histórico de alterações.

A versão canônica ainda é considerada preliminar neste ponto, porque a renderização produzida pelo núcleo formal será confirmada no Bloco 5. O arquivo será sincronizado novamente depois que todas as referências forem analisadas e reconstruídas pela AST.

### Barreiras de integridade

O bloco verifica:

- quantidade total de registros;
- unicidade dos IDs;
- presença de todos os campos obrigatórios;
- correspondência entre fonte, corpus canônico e changelog;
- ausência de alterações não documentadas.

### Resultado esperado

Ao final, o CAMPI possui uma representação tabular rastreável e pronta para ser submetida à gramática. A fonte original continua preservada separadamente.

In [3]:
# ----------------------------------------------------------
# 3.1 Função de padronização segura de espaços
# ----------------------------------------------------------

def normalizar_espacos_fora_de_strings(texto: str) -> str:
    """Padroniza somente os espaços externos aos valores entre aspas simples."""
    if not isinstance(texto, str):
        raise TypeError("A expressão Nile deve ser uma string.")

    resultado = []
    espaco_pendente = False
    dentro_string = False

    for caractere in texto.strip():
        if caractere == "'":
            if not dentro_string and espaco_pendente:
                if resultado and resultado[-1] not in " (":
                    resultado.append(" ")
                espaco_pendente = False

            resultado.append(caractere)
            dentro_string = not dentro_string
            continue

        if dentro_string:
            resultado.append(caractere)
            continue

        if caractere.isspace():
            espaco_pendente = True
            continue

        if caractere == "(":
            while resultado and resultado[-1] == " ":
                resultado.pop()
            resultado.append(caractere)
            espaco_pendente = False
            continue

        if caractere == ")":
            while resultado and resultado[-1] == " ":
                resultado.pop()
            resultado.append(caractere)
            espaco_pendente = False
            continue

        if caractere == ",":
            while resultado and resultado[-1] == " ":
                resultado.pop()
            resultado.append(caractere)
            espaco_pendente = True
            continue

        if espaco_pendente and resultado and resultado[-1] not in " (":
            resultado.append(" ")

        espaco_pendente = False
        resultado.append(caractere)

    if dentro_string:
        raise ValueError("Foi encontrada uma string Nile sem fechamento de aspas.")

    return "".join(resultado).strip()


# Teste unitário da preservação de espaços dentro de valores literais.
_exemplo_literal = "define intent x: for endpoint('A  B') block protocol('ftp')"
if "'A  B'" not in normalizar_espacos_fora_de_strings(_exemplo_literal):
    raise AssertionError("A padronização alterou espaços dentro de um valor Nile.")


# ----------------------------------------------------------
# 3.2 Correções formais explícitas e rastreáveis
# ----------------------------------------------------------

CORRECOES_EXPLICITAS = {
    "campi_001": {
        "tipo": "reordenacao_sintatica",
        "descricao": (
            "O escopo for group('students') foi movido para antes da operação add, "
            "sem alteração dos componentes da intenção."
        ),
        "nile": (
            "define intent uniIntent: for group('students') "
            "add middlebox('copyright monitoring')"
        ),
    },
    "campi_036": {
        "tipo": "correcao_temporal",
        "descricao": (
            "A cláusula end('05:59') recebeu o construtor hour exigido pelo "
            "subconjunto temporal adotado: end hour('05:59')."
        ),
        "nile": None,
    },
}


# ----------------------------------------------------------
# 3.3 Construção da tabela canônica preliminar
# ----------------------------------------------------------

registros_canonicos = []
registros_changelog = []

for indice, registro in enumerate(intents, start=1):
    exemplo_id = f"campi_{indice:03d}"
    nile_original = registro["nile"]
    nile_preliminar = normalizar_espacos_fora_de_strings(nile_original)
    tipos_alteracao = []
    descricoes = []

    if exemplo_id == "campi_001":
        correcao = CORRECOES_EXPLICITAS[exemplo_id]
        nile_preliminar = correcao["nile"]
        tipos_alteracao.append(correcao["tipo"])
        descricoes.append(correcao["descricao"])

    if exemplo_id == "campi_036":
        correcao = CORRECOES_EXPLICITAS[exemplo_id]
        nile_preliminar = nile_preliminar.replace(
            "end('05:59')",
            "end hour('05:59')",
        )
        tipos_alteracao.append(correcao["tipo"])
        descricoes.append(correcao["descricao"])

    if nile_original != nile_original.strip():
        tipos_alteracao.append("remocao_espaco_externo")
        descricoes.append("Foram removidos espaços em branco nas extremidades da expressão.")

    if re.search(r"\b(?:endpoint|group|traffic|service|protocol|middlebox|quota|bandwidth|hour)\s+\(", nile_original):
        tipos_alteracao.append("padronizacao_espaco_funcao")
        descricoes.append("Foi removido o espaço entre o nome da função Nile e o parêntese de abertura.")

    tipos_alteracao = list(dict.fromkeys(tipos_alteracao))
    descricoes = list(dict.fromkeys(descricoes))

    registros_canonicos.append({
        "id": exemplo_id,
        "source_group": "CAMPI",
        "source_index": indice,
        "university": registro["university"].strip(),
        "nl": registro["text"].strip(),
        "nile_original": nile_original,
        "nile_canonical": nile_preliminar,
        "changed": nile_original != nile_preliminar,
        "change_types": " | ".join(tipos_alteracao),
        "change_description": " | ".join(descricoes),
        "review_status": "correcao_formal_documentada" if nile_original != nile_preliminar else "preservado",
    })

    if nile_original != nile_preliminar:
        registros_changelog.append({
            "id": exemplo_id,
            "nile_original": nile_original,
            "nile_canonical_preliminary": nile_preliminar,
            "change_types": " | ".join(tipos_alteracao),
            "change_description": " | ".join(descricoes),
        })


df_campi = pd.DataFrame(registros_canonicos)
df_changelog = pd.DataFrame(registros_changelog)


# ----------------------------------------------------------
# 3.4 Verificações de quantidade e unicidade
# ----------------------------------------------------------

if len(df_campi) != EXPECTED_EXAMPLES:
    raise RuntimeError("A base canônica não contém exatamente 50 registros.")

if df_campi["id"].duplicated().any():
    raise RuntimeError("Foram encontrados identificadores duplicados.")

if df_campi["nl"].duplicated().any():
    raise RuntimeError("Foram encontrados textos em linguagem natural duplicados.")


# ----------------------------------------------------------
# 3.5 Salvamento preliminar dos arquivos tabulares
# ----------------------------------------------------------

df_campi.to_csv(CAMPI_CANONICAL_PATH, index=False, encoding="utf-8")
df_changelog.to_csv(CAMPI_CHANGELOG_PATH, index=False, encoding="utf-8")


# ----------------------------------------------------------
# 3.6 Exibição da base operacional preliminar
# ----------------------------------------------------------

exibir_tabela(
    df_campi[[
        "id",
        "university",
        "nl",
        "nile_original",
        "nile_canonical",
        "changed",
        "review_status",
    ]],
    "CAMPI operacional antes da renderização canônica pela AST",
    altura_px=620,
)


# ----------------------------------------------------------
# 3.7 Exibição do registro de alterações
# ----------------------------------------------------------

exibir_tabela(
    df_changelog,
    "Alterações formais registradas no CAMPI",
    altura_px=420,
)


# ----------------------------------------------------------
# 3.8 Saída do bloco
# ----------------------------------------------------------

print("Bloco 3 concluído")
print(f"Registros canônicos: {len(df_campi)}")
print(f"Registros alterados nesta etapa: {len(df_changelog)}")
print("Status: OK")

ID,universidade,NL,Nile original,Nile canônica,alterada,status da revisão
campi_001,University of Illinois - Urbana Champaign,"If a student is in obvious violations of copyright law by using a room's wired connection (ie ResNet) to distribute copyrighted materials, the room's connection will be disabled, and the issue could be sent to Housing Student Judicial Affairs",define intent uniIntent: add middlebox('copyright monitoring') for group('students'),define intent uniIntent: for group('students') add middlebox('copyright monitoring'),sim,correção formal documentada
campi_002,University of Illinois - Urbana Champaign,"Currently, the University of Illinois does not have any rate limits",define intent uniIntent: for endpoint('university') unset bandwidth(),define intent uniIntent: for endpoint('university') unset bandwidth(),não,preservada
campi_003,University of Illinois - Urbana Champaign,University Housing monitors only the amount of traffic of each user,define intent uniIntent: for endpoint('dorms') add middlebox('traffic monitor'),define intent uniIntent: for endpoint('dorms') add middlebox('traffic monitor'),não,preservada
campi_004,University of Illinois - Urbana Champaign,CounterStrike server is blocked by the University firewall,define intent uniIntent: for endpoint('university') add middlebox('firewall') block service('CounterStrike'),define intent uniIntent: for endpoint('university') add middlebox('firewall') block service('CounterStrike'),não,preservada
campi_005,University of Illinois - Urbana Champaign,AIM chat and file transfering is allowed by the University firewall,"define intent uniIntent: for endpoint('university') add middlebox('firewall') allow service('AIM chat'), service('file transfer')","define intent uniIntent: for endpoint('university') add middlebox('firewall') allow service('AIM chat'), service('file transfer')",não,preservada
campi_006,University of Illinois - Urbana Champaign,Battlenet is allowed by the University firewall,define intent uniIntent: for endpoint('university') add middlebox('firewall') allow service('Battlenet'),define intent uniIntent: for endpoint('university') add middlebox('firewall') allow service('Battlenet'),não,preservada
campi_007,University of Illinois - Urbana Champaign,H323 video conferencing is allowed by the University firewall,define intent uniIntent: for endpoint('university') add middlebox('firewall') allow traffic('H323 video conferencing'),define intent uniIntent: for endpoint('university') add middlebox('firewall') allow traffic('H323 video conferencing'),não,preservada
campi_008,University of Illinois - Urbana Champaign,Everquest is blocked by the University firewall,define intent uniIntent: for endpoint('university') add middlebox('firewall') block service('Everquest'),define intent uniIntent: for endpoint('university') add middlebox('firewall') block service('Everquest'),não,preservada
campi_009,University of Illinois - Urbana Champaign,HTTP and HTTPS are allowed by the University firewall,"define intent uniIntent: for endpoint('university') add middlebox('firewall') allow protocol('HTTP'), protocol('HTTPS')","define intent uniIntent: for endpoint('university') add middlebox('firewall') allow protocol('HTTP'), protocol('HTTPS')",não,preservada
campi_010,University of Illinois - Urbana Champaign,IMAP and secure IMAP are allowed by the University firewall,"define intent uniIntent: for endpoint('university') add middlebox('firewall') allow protocol('IMAP'), protocol('secure IMAP')","define intent uniIntent: for endpoint('university') add middlebox('firewall') allow protocol('IMAP'), protocol('secure IMAP')",não,preservada


ID,Nile original,Nile canônica preliminar,tipos de alteração,descrição da alteração
campi_001,define intent uniIntent: add middlebox('copyright monitoring') for group('students'),define intent uniIntent: for group('students') add middlebox('copyright monitoring'),reordenação sintática,"O escopo for group('students') foi movido para antes da operação add, sem alteração dos componentes da intenção."
campi_036,"define intent uniIntent: for group('students') set quota('download', '5','gb/d') start hour('06:00') end('05:59')","define intent uniIntent: for group('students') set quota('download', '5', 'gb/d') start hour('06:00') end hour('05:59')",correção temporal,A cláusula end('05:59') recebeu o construtor hour exigido pelo subconjunto temporal adotado: end hour('05:59').
campi_038,"define intent uniIntent: for traffic ('peer2peer'), endpoint('campus') set bandwidth('max', '30', 'mbps')","define intent uniIntent: for traffic('peer2peer'), endpoint('campus') set bandwidth('max', '30', 'mbps')",padronização do espaço da função,Foi removido o espaço entre o nome da função Nile e o parêntese de abertura.
campi_040,"define intent uniIntent: for group('dorms') set quota('any', '200', 'gb/wk')","define intent uniIntent: for group('dorms') set quota('any', '200', 'gb/wk')",remoção de espaço externo,Foram removidos espaços em branco nas extremidades da expressão.
campi_041,define intent uniIntent: from endpoint('internet') to endpoint('network') block protocol('ftp'),define intent uniIntent: from endpoint('internet') to endpoint('network') block protocol('ftp'),remoção de espaço externo,Foram removidos espaços em branco nas extremidades da expressão.
campi_042,"define intent uniIntent: for endpoint('network') add middlebox('network border system'), middlebox('ips'), middlebox('firewall'), middlebox('unit firewall')","define intent uniIntent: for endpoint('network') add middlebox('network border system'), middlebox('ips'), middlebox('firewall'), middlebox('unit firewall')",remoção de espaço externo,Foram removidos espaços em branco nas extremidades da expressão.
campi_043,define intent uniIntent: for endpoint('network') add middlebox('traffic monitor'),define intent uniIntent: for endpoint('network') add middlebox('traffic monitor'),remoção de espaço externo,Foram removidos espaços em branco nas extremidades da expressão.
campi_045,define intent uniIntent: for endpoint('network') add middlebox('traffic monitor'),define intent uniIntent: for endpoint('network') add middlebox('traffic monitor'),remoção de espaço externo,Foram removidos espaços em branco nas extremidades da expressão.


Bloco 3 concluído
Registros canônicos: 50
Registros alterados nesta etapa: 8
Status: OK


## Bloco 4 - Consolidação da gramática do subconjunto Nile

### Objetivo

Este bloco formaliza, em Lark, o subconjunto da linguagem Nile necessário para representar os 50 exemplos do CAMPI. A gramática funciona como especificação operacional da sintaxe aceita pelo experimento e como entrada do parser implementado no bloco seguinte.

### Escopo da gramática

A gramática cobre os elementos observados no corpus, incluindo:

- declaração de intenção;
- escopos `for`, `from`, `to`, `route` e `route_for`, conforme aplicável;
- endpoints, grupos e direções de rota;
- operações `add`, `allow`, `block`, `set` e `unset`;
- middleboxes, protocolos, serviços, tráfego, banda e cotas;
- listas de targets e itens;
- valores, unidades e restrições temporais;
- literais entre aspas e identificadores do subconjunto.

A cobertura é deliberadamente restrita. Construções não utilizadas pelo CAMPI não são adicionadas apenas para aproximar uma gramática completa da Nile.

### Verificações textuais

Antes de salvar o arquivo, o bloco confere a presença das regras essenciais e a coerência dos símbolos utilizados. Essas verificações reduzem o risco de uma gramática incompleta ser propagada para o parser.

### Artefato produzido

A gramática é persistida em:

```text
nile_subset.lark
```

Esse arquivo é consumido por `nile_core.py`, pela F1 e pelos notebooks experimentais da F3.

### Interpretação metodológica

A aceitação pela gramática representa validade sintática dentro do subconjunto estudado. Ela não garante, por si só, que a estrutura resultante seja semanticamente coerente. Por isso, a F0 mantém separadas a análise sintática e a verificação estrutural estática.

### Resultado esperado

O bloco termina com um resumo das construções cobertas e confirma que o arquivo da gramática foi criado corretamente.

In [4]:
# ----------------------------------------------------------
# 4.1 Definição da gramática Lark do subconjunto CAMPI
# ----------------------------------------------------------

NILE_GRAMMAR = r"""start: "define" "intent" NAME ":" scope operation+ interval?

?scope: for_scope                              -> scope_for
      | route_scope for_scope?                 -> scope_route
route_scope: "from" endpoint_ref "to" endpoint_ref
for_scope: "for" target_ref ("," target_ref)*

?target_ref: endpoint_ref | group_ref | traffic_ref
endpoint_ref: "endpoint" "(" STRING ")"
group_ref: "group" "(" STRING ")"
traffic_ref: "traffic" "(" STRING ")"
service_ref: "service" "(" STRING ")"
protocol_ref: "protocol" "(" STRING ")"
middlebox_ref: "middlebox" "(" STRING ")"

?operation: add_op | allow_op | block_op | set_op | unset_op
add_op: "add" middlebox_ref ("," middlebox_ref)*
allow_op: "allow" match_ref ("," match_ref)*
block_op: "block" match_ref ("," match_ref)*
?match_ref: service_ref | traffic_ref | protocol_ref
set_op: "set" policy_spec ("," policy_spec)*
unset_op: "unset" bandwidth_empty ("," bandwidth_empty)*
?policy_spec: quota_spec | bandwidth_spec
quota_spec: "quota" "(" STRING "," STRING "," STRING ")"
bandwidth_spec: "bandwidth" "(" STRING "," STRING "," STRING ")"
bandwidth_empty: "bandwidth" "(" ")"

interval: start_clause end_clause
start_clause: "start" hour_ref
end_clause: "end" hour_ref
hour_ref: "hour" "(" STRING ")"

STRING: /'[^'\r\n]*'/
NAME: /[A-Za-z_][A-Za-z0-9_]*/

%import common.WS
%ignore WS
"""


# ----------------------------------------------------------
# 4.2 Verificação textual das regras essenciais
# ----------------------------------------------------------

regras_essenciais = [
    '"define" "intent"',
    'route_scope',
    'for_scope',
    'add_op',
    'allow_op',
    'block_op',
    'set_op',
    'unset_op',
    'interval',
]

faltantes = [regra for regra in regras_essenciais if regra not in NILE_GRAMMAR]
if faltantes:
    raise RuntimeError(f"Regras essenciais ausentes da gramática: {faltantes}")


# ----------------------------------------------------------
# 4.3 Salvamento da gramática como artefato independente
# ----------------------------------------------------------

GRAMMAR_PATH.write_text(NILE_GRAMMAR.strip() + "\n", encoding="utf-8")


# ----------------------------------------------------------
# 4.4 Resumo da cobertura gramatical
# ----------------------------------------------------------

cobertura_gramatical = pd.DataFrame([
    {"categoria": "Escopo", "construcoes": "for; from/to; from/to + for", "status": "OK"},
    {"categoria": "Alvos", "construcoes": "endpoint; group; traffic", "status": "OK"},
    {"categoria": "ACL", "construcoes": "allow; block; service; traffic; protocol", "status": "OK"},
    {"categoria": "Funções de rede", "construcoes": "add middlebox", "status": "OK"},
    {"categoria": "QoS", "construcoes": "set quota; set/unset bandwidth", "status": "OK"},
    {"categoria": "Temporal", "construcoes": "start hour; end hour", "status": "OK"},
    {"categoria": "Fora do subconjunto", "construcoes": "remove; date; datetime", "status": "NÃO ACEITO"},
])

exibir_tabela(
    cobertura_gramatical,
    "Cobertura da gramática consolidada do subconjunto Nile",
    altura_px=320,
)


# ----------------------------------------------------------
# 4.5 Saída do bloco
# ----------------------------------------------------------

print("Bloco 4 concluído")
print(f"Gramática salva em: {GRAMMAR_PATH}")
print(f"SHA-256: {calcular_sha256(GRAMMAR_PATH)}")
print("Status: OK")

categoria,construcoes,status
Escopo,for; from/to; from/to + for,OK
Alvos,endpoint; group; traffic,OK
ACL,allow; block; service; traffic; protocol,OK
Funções de rede,add middlebox,OK
QoS,set quota; set/unset bandwidth,OK
Temporal,start hour; end hour,OK
Fora do subconjunto,remove; date; datetime,NÃO ACEITO


Bloco 4 concluído
Gramática salva em: /kaggle/working/f0_operacional/nile_subset.lark
SHA-256: 427b02e12e25f79a15f8a34f1f97c9d7e383bd82ec63a1ab244a35b4d3311a5e
Status: OK


## Bloco 5 - Parser Lark, AST e verificação semântica estática

### Objetivo

Este bloco implementa o núcleo formal da F0. Ele transforma expressões Nile em uma Árvore de Sintaxe Abstrata (AST), verifica regras estruturais do subconjunto e permite reconstruir uma forma canônica da expressão.

### Componentes de `nile_core.py`

O módulo gerado reúne funções para:

- carregar a gramática Lark;
- analisar uma expressão Nile;
- transformar a árvore do parser em AST;
- representar escopos, targets, operações, itens e restrições temporais;
- verificar combinações estruturais permitidas;
- renderizar a AST novamente como Nile canônica;
- produzir diagnósticos para expressões inválidas;
- gerar assinaturas e atributos estruturais usados nas métricas e nos folds.

### Cadeia aplicada às referências

Cada uma das 50 referências preliminares passa pela sequência:

```text
Nile preliminar
      ↓
parser Lark
      ↓
AST
      ↓
verificação estrutural estática
      ↓
renderização canônica
      ↓
nova análise e comparação
```

A expressão renderizada é usada para consolidar a versão operacional da Nile. Quando a renderização altera apenas a forma textual, o changelog é sincronizado para preservar a rastreabilidade.

### Atualização do corpus

O bloco acrescenta à base canônica informações como:

- sucesso sintático;
- validade estrutural;
- tipo de escopo;
- quantidade de targets;
- quantidade e tipos de operações;
- presença de restrição temporal;
- hashes da expressão canônica.

Depois dessas verificações, `campi_canonical.csv` é salvo novamente como versão definitiva da F0.

### Limites

A verificação é estática e específica do subconjunto estudado. Ela não executa políticas de rede, não avalia implantação e não prova propriedades gerais da linguagem Nile.

### Resultado esperado

As 50 referências devem ser analisadas e renderizadas sem perda estrutural. O bloco produz `nile_core.py` e o perfil estrutural utilizado nos testes, métricas e folds.

In [5]:
# ----------------------------------------------------------
# 5.1 Conteúdo do módulo nile_core.py
# ----------------------------------------------------------

NILE_CORE_CODE = r"""from __future__ import annotations

import copy
import re
from collections import Counter
from typing import Any, Dict, Iterable, List, Optional

from lark import Lark, Transformer, UnexpectedInput


class NileASTTransformer(Transformer):
    def NAME(self, token):
        return str(token)

    def STRING(self, token):
        return str(token)[1:-1]

    def endpoint_ref(self, items):
        return {"kind": "endpoint", "value": items[0]}

    def group_ref(self, items):
        return {"kind": "group", "value": items[0]}

    def traffic_ref(self, items):
        return {"kind": "traffic", "value": items[0]}

    def service_ref(self, items):
        return {"kind": "service", "value": items[0]}

    def protocol_ref(self, items):
        return {"kind": "protocol", "value": items[0]}

    def middlebox_ref(self, items):
        return {"kind": "middlebox", "value": items[0]}

    def route_scope(self, items):
        return {"from": items[0], "to": items[1]}

    def for_scope(self, items):
        return {"targets": list(items)}

    def scope_for(self, items):
        return {
            "type": "for",
            "from": None,
            "to": None,
            "targets": items[0]["targets"],
        }

    def scope_route(self, items):
        route = items[0]
        targets = items[1]["targets"] if len(items) > 1 else []
        return {
            "type": "route_for" if targets else "route",
            "from": route["from"],
            "to": route["to"],
            "targets": targets,
        }

    def add_op(self, items):
        return {"operator": "add", "items": list(items)}

    def allow_op(self, items):
        return {"operator": "allow", "items": list(items)}

    def block_op(self, items):
        return {"operator": "block", "items": list(items)}

    def quota_spec(self, items):
        return {
            "kind": "quota",
            "constraint": items[0],
            "value": items[1],
            "unit": items[2],
        }

    def bandwidth_spec(self, items):
        return {
            "kind": "bandwidth",
            "constraint": items[0],
            "value": items[1],
            "unit": items[2],
        }

    def bandwidth_empty(self, _items):
        return {"kind": "bandwidth"}

    def set_op(self, items):
        return {"operator": "set", "items": list(items)}

    def unset_op(self, items):
        return {"operator": "unset", "items": list(items)}

    def hour_ref(self, items):
        return {"kind": "hour", "value": items[0]}

    def start_clause(self, items):
        return items[0]

    def end_clause(self, items):
        return items[0]

    def interval(self, items):
        return {"start": items[0], "end": items[1]}

    def start(self, items):
        intent_id = items[0]
        scope = items[1]
        operations = []
        interval = None

        for item in items[2:]:
            if isinstance(item, dict) and "operator" in item:
                operations.append(item)
            elif isinstance(item, dict) and "start" in item and "end" in item:
                interval = item

        return {
            "intent_id": intent_id,
            "scope": scope,
            "operations": operations,
            "interval": interval,
        }


class NileValidator:
    QUOTA_CONSTRAINTS = {"any", "download", "upload"}
    BANDWIDTH_CONSTRAINTS = {"min", "max"}
    QUOTA_UNITS = {"gb", "gb/d", "gb/wk", "gb/mth", "mb/h"}
    BANDWIDTH_UNITS = {"mbps"}
    ALLOWED_TARGET_KINDS = {"endpoint", "group", "traffic"}
    ALLOWED_MATCH_KINDS = {"service", "traffic", "protocol"}
    ALLOWED_OPERATORS = {"add", "allow", "block", "set", "unset"}

    def __init__(self, grammar_text: str):
        if not isinstance(grammar_text, str) or not grammar_text.strip():
            raise ValueError("A gramática Nile não pode estar vazia.")

        self.grammar_text = grammar_text
        self.parser = Lark(
            grammar_text,
            parser="lalr",
            start="start",
            maybe_placeholders=False,
            propagate_positions=True,
        )
        self.transformer = NileASTTransformer()

    @staticmethod
    def _syntax_error(exc: UnexpectedInput, text: str) -> Dict[str, Any]:
        context = ""
        try:
            context = exc.get_context(text, span=45).strip()
        except Exception:
            context = ""

        expected = sorted(getattr(exc, "expected", []) or [])
        allowed = sorted(getattr(exc, "allowed", []) or [])

        return {
            "type": exc.__class__.__name__,
            "message": str(exc).splitlines()[0],
            "line": getattr(exc, "line", None),
            "column": getattr(exc, "column", None),
            "context": context,
            "expected": expected or allowed,
        }

    def parse(self, text: str) -> Dict[str, Any]:
        if not isinstance(text, str):
            return {
                "syntax_valid": False,
                "ast": None,
                "syntax_errors": [{
                    "type": "TypeError",
                    "message": "A expressão Nile deve ser uma string.",
                    "line": None,
                    "column": None,
                    "context": "",
                    "expected": [],
                }],
            }

        if not text.strip():
            return {
                "syntax_valid": False,
                "ast": None,
                "syntax_errors": [{
                    "type": "EmptyInput",
                    "message": "A expressão Nile está vazia.",
                    "line": 1,
                    "column": 1,
                    "context": "",
                    "expected": ["define"],
                }],
            }

        try:
            tree = self.parser.parse(text)
            ast = self.transformer.transform(tree)
            return {
                "syntax_valid": True,
                "ast": ast,
                "syntax_errors": [],
            }
        except UnexpectedInput as exc:
            return {
                "syntax_valid": False,
                "ast": None,
                "syntax_errors": [self._syntax_error(exc, text)],
            }
        except Exception as exc:
            return {
                "syntax_valid": False,
                "ast": None,
                "syntax_errors": [{
                    "type": exc.__class__.__name__,
                    "message": str(exc),
                    "line": None,
                    "column": None,
                    "context": "",
                    "expected": [],
                }],
            }

    @staticmethod
    def _is_nonempty_string(value: Any) -> bool:
        return isinstance(value, str) and bool(value.strip())

    @staticmethod
    def _is_number_string(value: Any) -> bool:
        return isinstance(value, str) and bool(re.fullmatch(r"(?:0|[1-9]\d*)(?:\.\d+)?", value.strip()))

    @staticmethod
    def _valid_hour(value: Any) -> bool:
        if not isinstance(value, str) or not re.fullmatch(r"\d{2}:\d{2}", value):
            return False
        hour, minute = map(int, value.split(":"))
        return 0 <= hour <= 23 and 0 <= minute <= 59

    @staticmethod
    def _add_error(errors: List[Dict[str, str]], code: str, path: str, message: str) -> None:
        errors.append({"code": code, "path": path, "message": message})

    def validate_ast(self, ast: Optional[Dict[str, Any]]) -> Dict[str, Any]:
        errors: List[Dict[str, str]] = []

        if not isinstance(ast, dict):
            self._add_error(errors, "AST_MISSING", "$", "A AST não foi produzida.")
            return {"structural_valid": False, "structural_errors": errors}

        intent_id = ast.get("intent_id")
        if not self._is_nonempty_string(intent_id):
            self._add_error(errors, "INTENT_ID_EMPTY", "$.intent_id", "O identificador da intenção está vazio.")

        scope = ast.get("scope")
        if not isinstance(scope, dict):
            self._add_error(errors, "SCOPE_MISSING", "$.scope", "O escopo da intenção está ausente.")
        else:
            scope_type = scope.get("type")
            targets = scope.get("targets", [])
            origin = scope.get("from")
            destination = scope.get("to")

            if scope_type not in {"for", "route", "route_for"}:
                self._add_error(errors, "SCOPE_TYPE_INVALID", "$.scope.type", "Tipo de escopo não reconhecido.")

            if scope_type == "for" and not targets:
                self._add_error(errors, "TARGET_REQUIRED", "$.scope.targets", "O escopo for exige pelo menos um alvo.")

            if scope_type in {"route", "route_for"}:
                for name, value in (("from", origin), ("to", destination)):
                    if not isinstance(value, dict) or value.get("kind") != "endpoint" or not self._is_nonempty_string(value.get("value")):
                        self._add_error(errors, "ROUTE_ENDPOINT_INVALID", f"$.scope.{name}", f"O campo {name} deve ser um endpoint não vazio.")

            if scope_type == "route" and targets:
                self._add_error(errors, "ROUTE_TARGETS_UNEXPECTED", "$.scope.targets", "O escopo route não deve conter alvos adicionais.")

            for index, target in enumerate(targets):
                path = f"$.scope.targets[{index}]"
                if not isinstance(target, dict):
                    self._add_error(errors, "TARGET_INVALID", path, "O alvo deve ser um objeto estruturado.")
                    continue
                if target.get("kind") not in self.ALLOWED_TARGET_KINDS:
                    self._add_error(errors, "TARGET_KIND_INVALID", f"{path}.kind", "Tipo de alvo fora do subconjunto CAMPI.")
                if not self._is_nonempty_string(target.get("value")):
                    self._add_error(errors, "TARGET_VALUE_EMPTY", f"{path}.value", "O valor do alvo está vazio.")

        operations = ast.get("operations")
        if not isinstance(operations, list) or not operations:
            self._add_error(errors, "OPERATION_REQUIRED", "$.operations", "A intenção exige pelo menos uma operação.")
        else:
            for op_index, operation in enumerate(operations):
                op_path = f"$.operations[{op_index}]"
                if not isinstance(operation, dict):
                    self._add_error(errors, "OPERATION_INVALID", op_path, "A operação deve ser um objeto estruturado.")
                    continue

                operator = operation.get("operator")
                items = operation.get("items")

                if operator not in self.ALLOWED_OPERATORS:
                    self._add_error(errors, "OPERATOR_INVALID", f"{op_path}.operator", "Operador fora do subconjunto CAMPI.")

                if not isinstance(items, list) or not items:
                    self._add_error(errors, "OPERATION_ITEMS_REQUIRED", f"{op_path}.items", "A operação exige pelo menos um item.")
                    continue

                for item_index, item in enumerate(items):
                    item_path = f"{op_path}.items[{item_index}]"
                    if not isinstance(item, dict):
                        self._add_error(errors, "ITEM_INVALID", item_path, "O item deve ser um objeto estruturado.")
                        continue

                    kind = item.get("kind")

                    if operator == "add":
                        if kind != "middlebox":
                            self._add_error(errors, "ADD_KIND_INVALID", f"{item_path}.kind", "A operação add aceita apenas middlebox.")
                        if not self._is_nonempty_string(item.get("value")):
                            self._add_error(errors, "MIDDLEBOX_VALUE_EMPTY", f"{item_path}.value", "O middlebox não pode ser vazio.")

                    elif operator in {"allow", "block"}:
                        if kind not in self.ALLOWED_MATCH_KINDS:
                            self._add_error(errors, "MATCH_KIND_INVALID", f"{item_path}.kind", "allow/block aceitam service, traffic ou protocol.")
                        if not self._is_nonempty_string(item.get("value")):
                            self._add_error(errors, "MATCH_VALUE_EMPTY", f"{item_path}.value", "O item de allow/block não pode ser vazio.")

                    elif operator == "set":
                        constraint = item.get("constraint")
                        value = item.get("value")
                        unit = item.get("unit")

                        if kind == "quota":
                            if constraint not in self.QUOTA_CONSTRAINTS:
                                self._add_error(errors, "QUOTA_CONSTRAINT_INVALID", f"{item_path}.constraint", "Restrição de quota inválida.")
                            if unit not in self.QUOTA_UNITS:
                                self._add_error(errors, "QUOTA_UNIT_INVALID", f"{item_path}.unit", "Unidade de quota inválida para o subconjunto CAMPI.")
                        elif kind == "bandwidth":
                            if constraint not in self.BANDWIDTH_CONSTRAINTS:
                                self._add_error(errors, "BANDWIDTH_CONSTRAINT_INVALID", f"{item_path}.constraint", "Restrição de bandwidth inválida.")
                            if unit not in self.BANDWIDTH_UNITS:
                                self._add_error(errors, "BANDWIDTH_UNIT_INVALID", f"{item_path}.unit", "Unidade de bandwidth inválida para o subconjunto CAMPI.")
                        else:
                            self._add_error(errors, "SET_KIND_INVALID", f"{item_path}.kind", "A operação set aceita apenas quota ou bandwidth.")

                        if not self._is_number_string(value):
                            self._add_error(errors, "POLICY_VALUE_INVALID", f"{item_path}.value", "O valor da política deve ser numérico e não negativo.")

                    elif operator == "unset":
                        if kind != "bandwidth":
                            self._add_error(errors, "UNSET_KIND_INVALID", f"{item_path}.kind", "A operação unset aceita apenas bandwidth() no subconjunto CAMPI.")
                        unexpected = sorted(set(item) - {"kind"})
                        if unexpected:
                            self._add_error(errors, "UNSET_ARGUMENTS_UNEXPECTED", item_path, "A operação unset não recebe argumentos no subconjunto adotado.")

        interval = ast.get("interval")
        if interval is not None:
            if not isinstance(interval, dict):
                self._add_error(errors, "INTERVAL_INVALID", "$.interval", "O intervalo deve ser um objeto estruturado.")
            else:
                for name in ("start", "end"):
                    ref = interval.get(name)
                    path = f"$.interval.{name}"
                    if not isinstance(ref, dict) or ref.get("kind") != "hour":
                        self._add_error(errors, "INTERVAL_KIND_INVALID", path, "O subconjunto CAMPI aceita intervalos do tipo hour.")
                    elif not self._valid_hour(ref.get("value")):
                        self._add_error(errors, "HOUR_INVALID", f"{path}.value", "A hora deve seguir HH:MM entre 00:00 e 23:59.")

        return {
            "structural_valid": len(errors) == 0,
            "structural_errors": errors,
        }

    def validate(self, text: str) -> Dict[str, Any]:
        parsed = self.parse(text)
        if not parsed["syntax_valid"]:
            return {
                **parsed,
                "structural_valid": False,
                "structural_errors": [],
                "valid": False,
                "canonical_nile": None,
                "feedback": self.format_feedback(parsed["syntax_errors"], []),
            }

        structural = self.validate_ast(parsed["ast"])
        canonical_nile = render_ast(parsed["ast"])
        valid = parsed["syntax_valid"] and structural["structural_valid"]

        return {
            **parsed,
            **structural,
            "valid": valid,
            "canonical_nile": canonical_nile,
            "feedback": self.format_feedback(parsed["syntax_errors"], structural["structural_errors"]),
        }

    @staticmethod
    def format_feedback(syntax_errors: Iterable[Dict[str, Any]], structural_errors: Iterable[Dict[str, Any]]) -> str:
        syntax_errors = list(syntax_errors)
        structural_errors = list(structural_errors)

        if not syntax_errors and not structural_errors:
            return "Expressão Nile válida."

        lines = []
        for error in syntax_errors:
            location = ""
            if error.get("line") is not None and error.get("column") is not None:
                location = f" na linha {error['line']}, coluna {error['column']}"
            lines.append(f"ERRO SINTÁTICO{location}: {error.get('message', '')}")
            if error.get("context"):
                lines.append(f"Contexto: {error['context']}")

        for error in structural_errors:
            lines.append(f"ERRO ESTRUTURAL [{error.get('code')} em {error.get('path')}]: {error.get('message')}")

        return "\n".join(lines)


def _quote(value: str) -> str:
    if "'" in value:
        raise ValueError("O subconjunto adotado não permite apóstrofo dentro de valores Nile.")
    return f"'{value}'"


def _render_ref(item: Dict[str, Any]) -> str:
    return f"{item['kind']}({_quote(item['value'])})"


def render_ast(ast: Dict[str, Any]) -> str:
    if not isinstance(ast, dict):
        raise TypeError("A AST deve ser um dicionário.")

    parts = [f"define intent {ast['intent_id']}:"]
    scope = ast["scope"]

    if scope["type"] in {"route", "route_for"}:
        parts.append(f"from {_render_ref(scope['from'])}")
        parts.append(f"to {_render_ref(scope['to'])}")

    if scope["type"] in {"for", "route_for"}:
        targets = ", ".join(_render_ref(item) for item in scope["targets"])
        parts.append(f"for {targets}")

    for operation in ast["operations"]:
        operator = operation["operator"]
        rendered_items = []

        for item in operation["items"]:
            if operator in {"add", "allow", "block"}:
                rendered_items.append(_render_ref(item))
            elif operator == "set":
                rendered_items.append(
                    f"{item['kind']}({_quote(item['constraint'])}, {_quote(item['value'])}, {_quote(item['unit'])})"
                )
            elif operator == "unset":
                rendered_items.append(f"{item['kind']}()")
            else:
                raise ValueError(f"Operador não reconhecido: {operator}")

        parts.append(f"{operator} {', '.join(rendered_items)}")

    interval = ast.get("interval")
    if interval is not None:
        parts.append(f"start hour({_quote(interval['start']['value'])})")
        parts.append(f"end hour({_quote(interval['end']['value'])})")

    return " ".join(parts)


def structural_family(ast: Dict[str, Any]) -> str:
    families = set()
    for operation in ast.get("operations", []):
        operator = operation.get("operator")
        if operator == "add":
            families.add("middlebox")
        elif operator in {"allow", "block"}:
            families.add("acl")
        elif operator in {"set", "unset"}:
            families.add("qos")

    if len(families) > 1:
        return "mixed"
    if families:
        return next(iter(families))
    return "unknown"


def ast_complexity(ast: Dict[str, Any]) -> Dict[str, int]:
    scope = ast.get("scope", {})
    operations = ast.get("operations", [])
    n_targets = len(scope.get("targets", []) or [])
    n_items = sum(len(op.get("items", []) or []) for op in operations)
    has_route = int(scope.get("type") in {"route", "route_for"})
    has_temporal = int(ast.get("interval") is not None)
    score = n_targets + n_items + len(operations) + has_route + has_temporal
    return {
        "n_targets": n_targets,
        "n_operations": len(operations),
        "n_items": n_items,
        "has_route": has_route,
        "has_temporal": has_temporal,
        "complexity_score": score,
    }
"""


# ----------------------------------------------------------
# 5.2 Salvamento e compilação do módulo
# ----------------------------------------------------------

NILE_CORE_PATH.write_text(NILE_CORE_CODE.strip() + "\n", encoding="utf-8")
py_compile.compile(str(NILE_CORE_PATH), doraise=True)


# ----------------------------------------------------------
# 5.3 Importação controlada do módulo recém-gerado
# ----------------------------------------------------------

if str(F0_DIR) not in sys.path:
    sys.path.insert(0, str(F0_DIR))

importlib.invalidate_caches()

if "nile_core" in sys.modules:
    del sys.modules["nile_core"]

from nile_core import NileValidator, ast_complexity, render_ast, structural_family

validator = NileValidator(GRAMMAR_PATH.read_text(encoding="utf-8"))


# ----------------------------------------------------------
# 5.4 Análise das 50 referências preliminares
# ----------------------------------------------------------

validacoes_preliminares = []
niles_renderizadas = []

for linha in df_campi.itertuples(index=False):
    resultado = validator.validate(linha.nile_canonical)

    validacoes_preliminares.append({
        "id": linha.id,
        "syntax_valid": resultado["syntax_valid"],
        "structural_valid": resultado["structural_valid"],
        "valid": resultado["valid"],
        "feedback": resultado["feedback"],
    })

    if not resultado["valid"]:
        raise RuntimeError(
            f"A referência {linha.id} não foi aceita pelo validador:\n"
            f"{resultado['feedback']}"
        )

    niles_renderizadas.append(resultado["canonical_nile"])


df_campi["nile_canonical"] = niles_renderizadas


# ----------------------------------------------------------
# 5.5 Inclusão de atributos estruturais na base operacional
# ----------------------------------------------------------

familias = []
has_temporal = []
has_route = []
n_operations = []
n_items = []
complexity_scores = []

for nile in df_campi["nile_canonical"]:
    resultado = validator.validate(nile)
    ast_nile = resultado["ast"]
    complexidade = ast_complexity(ast_nile)

    familias.append(structural_family(ast_nile))
    has_temporal.append(bool(complexidade["has_temporal"]))
    has_route.append(bool(complexidade["has_route"]))
    n_operations.append(complexidade["n_operations"])
    n_items.append(complexidade["n_items"])
    complexity_scores.append(complexidade["complexity_score"])


df_campi["primary_family"] = familias
df_campi["has_temporal"] = has_temporal
df_campi["has_route"] = has_route
df_campi["n_operations"] = n_operations
df_campi["n_items"] = n_items
df_campi["complexity_score"] = complexity_scores


# ----------------------------------------------------------
# 5.6 Atualização do changelog após a renderização canônica
# ----------------------------------------------------------

registros_changelog_finais = []

for linha in df_campi.itertuples(index=False):
    if linha.nile_original == linha.nile_canonical:
        continue

    tipos = [item.strip() for item in str(linha.change_types).split("|") if item.strip()]
    descricoes = [item.strip() for item in str(linha.change_description).split("|") if item.strip()]

    if normalizar_espacos_fora_de_strings(linha.nile_original) != linha.nile_canonical:
        if "renderizacao_ast" not in tipos:
            tipos.append("renderizacao_ast")
            descricoes.append(
                "A expressão foi renderizada deterministicamente a partir da AST para padronizar vírgulas e espaços."
            )

    registros_changelog_finais.append({
        "id": linha.id,
        "nile_original": linha.nile_original,
        "nile_canonical": linha.nile_canonical,
        "change_types": " | ".join(dict.fromkeys(tipos)),
        "change_description": " | ".join(dict.fromkeys(descricoes)),
    })


df_changelog = pd.DataFrame(registros_changelog_finais)


# ----------------------------------------------------------
# 5.7 Sincronização dos metadados da base canônica
# ----------------------------------------------------------

mapa_alteracoes = {
    registro["id"]: {
        "change_types": registro["change_types"],
        "change_description": registro["change_description"],
    }
    for registro in registros_changelog_finais
}

df_campi["changed"] = df_campi["nile_original"] != df_campi["nile_canonical"]
df_campi["change_types"] = df_campi["id"].map(
    lambda exemplo_id: mapa_alteracoes.get(exemplo_id, {}).get("change_types", "")
)
df_campi["change_description"] = df_campi["id"].map(
    lambda exemplo_id: mapa_alteracoes.get(exemplo_id, {}).get("change_description", "")
)
df_campi["review_status"] = df_campi["changed"].map(
    {True: "correcao_formal_documentada", False: "preservado"}
)

if int(df_campi["changed"].sum()) != len(df_changelog):
    raise AssertionError(
        "A quantidade de registros alterados diverge do changelog final."
    )

if (
    df_campi.loc[df_campi["changed"], "change_types"]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):
    raise AssertionError(
        "Existe referência alterada sem tipo de alteração documentado."
    )


# ----------------------------------------------------------
# 5.8 Salvamento definitivo da base canônica
# ----------------------------------------------------------

df_campi.to_csv(CAMPI_CANONICAL_PATH, index=False, encoding="utf-8")
df_changelog.to_csv(CAMPI_CHANGELOG_PATH, index=False, encoding="utf-8")


# ----------------------------------------------------------
# 5.9 Tabela do perfil estrutural do CAMPI
# ----------------------------------------------------------

perfil_estrutural = (
    df_campi
    .groupby(["primary_family", "has_temporal"], dropna=False)
    .agg(
        exemplos=("id", "count"),
        media_operacoes=("n_operations", "mean"),
        media_itens=("n_items", "mean"),
        media_complexidade=("complexity_score", "mean"),
    )
    .reset_index()
)

for coluna in ["media_operacoes", "media_itens", "media_complexidade"]:
    perfil_estrutural[coluna] = perfil_estrutural[coluna].round(2)

exibir_tabela(
    perfil_estrutural,
    "Perfil estrutural das referências canônicas do CAMPI",
    altura_px=360,
)


# ----------------------------------------------------------
# 5.10 Saída do bloco
# ----------------------------------------------------------

print("Bloco 5 concluído")
print(f"Módulo gerado: {NILE_CORE_PATH}")
print(f"Referências processadas: {len(df_campi)}")
print(f"Referências válidas: {sum(item['valid'] for item in validacoes_preliminares)}")
print("Status: OK")

família estrutural,possui restrição temporal,exemplos,média de operações,média de itens,média de complexidade
ACL,não,8,1.00,2.00,4.38
middlebox,não,8,1.00,1.38,3.38
mista,não,18,2.06,2.50,5.56
QoS,não,13,1.00,1.15,3.23
QoS,sim,3,1.00,1.00,4.33


Bloco 5 concluído
Módulo gerado: /kaggle/working/f0_operacional/nile_core.py
Referências processadas: 50
Referências válidas: 50
Status: OK


## Bloco 6 - Testes do validador e validação das referências

### Objetivo

Este bloco testa o comportamento do núcleo formal e confirma que todas as referências canônicas do CAMPI são aceitas pelo parser e pelo verificador estrutural.

### Casos positivos

Os casos positivos incluem as 50 referências canônicas. Cada uma deve:

- ser aceita pela gramática;
- produzir uma AST;
- passar pelas regras estruturais;
- ser renderizada e novamente analisada;
- preservar os valores literais relevantes.

### Casos negativos dirigidos

O bloco constrói expressões inválidas para representar classes específicas de erro, como:

- sintaxe incompleta;
- operador ausente ou incompatível;
- escopo malformado;
- item sem argumentos obrigatórios;
- unidade inválida;
- intervalo temporal inconsistente;
- estrutura não coberta pelo subconjunto.

Esses casos verificam se o validador rejeita erros conhecidos e produz diagnósticos úteis. Eles funcionam como testes de regressão, não como amostra estatística nem como prova formal de correção para todas as combinações possíveis.

### Preservação de literais

Também são testados nomes de serviços, grupos, protocolos, middleboxes e unidades para assegurar que o parser e o renderer não traduzam, normalizem semanticamente ou substituam valores do CAMPI.

### Artefatos produzidos

O bloco salva:

- `validator_tests.jsonl`, com casos positivos e negativos;
- `validation_references.csv`, com o resultado das 50 referências.

### PSR de referência

Como as 50 expressões canônicas devem ser sintaticamente válidas, a referência esperada é:

```text
50/50 referências aceitas
PSR = 1
```

A validade estrutural é registrada separadamente da PSR.

### Resultado esperado

Todos os casos positivos devem passar e todos os casos negativos dirigidos devem ser rejeitados conforme a expectativa registrada.

In [6]:
# ----------------------------------------------------------
# 6.1 Construção dos casos positivos
# ----------------------------------------------------------

casos_positivos = []

for linha in df_campi.itertuples(index=False):
    casos_positivos.append({
        "test_id": f"positive_{linha.id}",
        "category": "positive_reference",
        "nile": linha.nile_canonical,
        "expected_syntax_valid": True,
        "expected_structural_valid": True,
        "expected_valid": True,
    })


# ----------------------------------------------------------
# 6.2 Construção dos casos negativos
# ----------------------------------------------------------

casos_negativos = [
    {
        "test_id": "negative_001",
        "category": "texto_invalido",
        "nile": "nonsense",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_002",
        "category": "caractere_extra",
        "nile": "define intent uniIntent: for endpoint('network') block protocol('ftp');",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_003",
        "category": "sem_identificador",
        "nile": "define intent : for endpoint('network') block protocol('ftp')",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_004",
        "category": "sem_dois_pontos",
        "nile": "define intent uniIntent for endpoint('network') block protocol('ftp')",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_005",
        "category": "sem_escopo",
        "nile": "define intent uniIntent: block protocol('ftp')",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_006",
        "category": "sem_operacao",
        "nile": "define intent uniIntent: for endpoint('network')",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_007",
        "category": "from_sem_to",
        "nile": "define intent uniIntent: from endpoint('internet') block protocol('ftp')",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_008",
        "category": "allow_vazio",
        "nile": "define intent uniIntent: for endpoint('network') allow",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_009",
        "category": "block_vazio",
        "nile": "define intent uniIntent: for endpoint('network') block",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_010",
        "category": "add_vazio",
        "nile": "define intent uniIntent: for endpoint('network') add",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_011",
        "category": "set_sem_argumentos",
        "nile": "define intent uniIntent: for endpoint('network') set bandwidth()",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_012",
        "category": "unset_com_argumentos",
        "nile": "define intent uniIntent: for endpoint('network') unset bandwidth('max', '4', 'mbps')",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_013",
        "category": "intervalo_incompleto",
        "nile": "define intent uniIntent: for group('students') set quota('download', '10', 'gb/d') start hour('10:00')",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_014",
        "category": "virgula_final",
        "nile": "define intent uniIntent: for endpoint('network') block protocol('ftp'),",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_015",
        "category": "texto_apos_intencao",
        "nile": "define intent uniIntent: for endpoint('network') block protocol('ftp') extra",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_016",
        "category": "ordem_invalida",
        "nile": "define intent uniIntent: add middlebox('firewall') for endpoint('network')",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_017",
        "category": "operacao_fora_subconjunto",
        "nile": "define intent uniIntent: for endpoint('network') remove middlebox('firewall')",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_018",
        "category": "target_sem_aspas",
        "nile": "define intent uniIntent: for endpoint(network) block protocol('ftp')",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_019",
        "category": "target_vazio",
        "nile": "define intent uniIntent: for endpoint('') block protocol('ftp')",
        "expected_syntax_valid": True,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_020",
        "category": "valor_nao_numerico",
        "nile": "define intent uniIntent: for endpoint('network') set bandwidth('max', 'abc', 'mbps')",
        "expected_syntax_valid": True,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_021",
        "category": "constraint_invalida",
        "nile": "define intent uniIntent: for endpoint('network') set bandwidth('download', '4', 'mbps')",
        "expected_syntax_valid": True,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_022",
        "category": "unidade_incompativel",
        "nile": "define intent uniIntent: for group('students') set quota('download', '10', 'mbps')",
        "expected_syntax_valid": True,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_023",
        "category": "hora_invalida",
        "nile": "define intent uniIntent: for group('students') set quota('download', '10', 'gb/d') start hour('25:00') end hour('26:00')",
        "expected_syntax_valid": True,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_024",
        "category": "quebra_de_linha_em_string",
        "nile": "define intent uniIntent: for endpoint('net\nwork') block protocol('ftp')",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },
    {
        "test_id": "negative_025",
        "category": "unset_quota_fora_subconjunto",
        "nile": "define intent uniIntent: for endpoint('network') unset quota()",
        "expected_syntax_valid": False,
        "expected_structural_valid": False,
        "expected_valid": False,
    },

]


# ----------------------------------------------------------
# 6.3 Teste de preservação de valores literais
# ----------------------------------------------------------

expressao_com_espaco_literal = (
    "define intent uniIntent: for endpoint('A  B') block protocol('ftp')"
)
resultado_espaco_literal = validator.validate(expressao_com_espaco_literal)

if not resultado_espaco_literal["valid"]:
    raise AssertionError("Uma expressão válida com espaço interno no literal foi rejeitada.")

if "'A  B'" not in resultado_espaco_literal["canonical_nile"]:
    raise AssertionError("A análise ou a renderização alterou o conteúdo do literal 'A  B'.")


# ----------------------------------------------------------
# 6.4 Execução e conferência dos testes
# ----------------------------------------------------------

casos_teste = casos_positivos + casos_negativos
resultados_testes = []

for caso in casos_teste:
    resultado = validator.validate(caso["nile"])

    passou = (
        resultado["syntax_valid"] == caso["expected_syntax_valid"]
        and resultado["structural_valid"] == caso["expected_structural_valid"]
        and resultado["valid"] == caso["expected_valid"]
    )

    resultados_testes.append({
        **caso,
        "observed_syntax_valid": resultado["syntax_valid"],
        "observed_structural_valid": resultado["structural_valid"],
        "observed_valid": resultado["valid"],
        "passed": passou,
        "feedback": resultado["feedback"],
    })


df_testes = pd.DataFrame(resultados_testes)

if not df_testes["passed"].all():
    falhas = df_testes.loc[~df_testes["passed"]]
    raise AssertionError(
        "Existem testes do validador com resultado divergente:\n"
        + falhas.to_string(index=False)
    )

salvar_jsonl(casos_teste, VALIDATOR_TESTS_PATH)


# ----------------------------------------------------------
# 6.4 Validação final das referências
# ----------------------------------------------------------

registros_validacao = []

for linha in df_campi.itertuples(index=False):
    resultado = validator.validate(linha.nile_canonical)

    roundtrip_ok = False
    if resultado["valid"]:
        segundo = validator.validate(resultado["canonical_nile"])
        roundtrip_ok = (
            segundo["valid"]
            and segundo["ast"] == resultado["ast"]
            and segundo["canonical_nile"] == resultado["canonical_nile"]
        )

    registros_validacao.append({
        "id": linha.id,
        "syntax_valid": resultado["syntax_valid"],
        "structural_valid": resultado["structural_valid"],
        "valid": resultado["valid"],
        "roundtrip_ok": roundtrip_ok,
        "syntax_error_count": len(resultado["syntax_errors"]),
        "structural_error_count": len(resultado["structural_errors"]),
        "feedback": resultado["feedback"],
    })


df_validacao_referencias = pd.DataFrame(registros_validacao)
df_validacao_referencias.to_csv(
    VALIDATION_REFERENCES_PATH,
    index=False,
    encoding="utf-8",
)

psr_referencia = df_validacao_referencias["syntax_valid"].mean()
taxa_estrutural_referencia = df_validacao_referencias["structural_valid"].mean()

if psr_referencia != 1.0:
    raise AssertionError(f"PSR de referência diferente de 1: {psr_referencia}")

if taxa_estrutural_referencia != 1.0:
    raise AssertionError(
        "Nem todas as referências foram consideradas estruturalmente completas."
    )

if not df_validacao_referencias["roundtrip_ok"].all():
    raise AssertionError("Alguma referência falhou no teste AST → Nile → AST.")


# ----------------------------------------------------------
# 6.5 Tabela de resumo dos testes
# ----------------------------------------------------------

resumo_testes = (
    df_testes
    .groupby(["category", "expected_valid"], dropna=False)
    .agg(
        casos=("test_id", "count"),
        aprovados=("passed", "sum"),
    )
    .reset_index()
)
resumo_testes["status"] = resumo_testes.apply(
    lambda linha: "OK" if linha["casos"] == linha["aprovados"] else "ERRO",
    axis=1,
)

exibir_tabela(
    resumo_testes,
    "Resultado dos testes positivos e negativos do validador",
    altura_px=520,
)


# ----------------------------------------------------------
# 6.6 Tabela da validação das referências
# ----------------------------------------------------------

exibir_tabela(
    df_validacao_referencias,
    "Validação sintática, estrutural e de ida e volta das 50 referências",
    altura_px=520,
)


# ----------------------------------------------------------
# 6.7 Saída do bloco
# ----------------------------------------------------------

print("Bloco 6 concluído")
print(f"Casos positivos: {len(casos_positivos)}")
print(f"Casos negativos: {len(casos_negativos)}")
print(f"PSR de referência: {psr_referencia:.4f}")
print(f"Taxa estrutural de referência: {taxa_estrutural_referencia:.4f}")
print("Status: OK")

categoria,resultado esperado,casos,aprovados,status
add_vazio,não,1,1,OK
allow_vazio,não,1,1,OK
block_vazio,não,1,1,OK
caractere_extra,não,1,1,OK
constraint_invalida,não,1,1,OK
from_sem_to,não,1,1,OK
hora_invalida,não,1,1,OK
intervalo_incompleto,não,1,1,OK
operacao_fora_subconjunto,não,1,1,OK
ordem_invalida,não,1,1,OK


ID,sintaxe válida,estrutura válida,expressão válida,ida e volta válida,erros sintáticos,erros estruturais,diagnóstico
campi_001,sim,sim,sim,sim,0,0,Expressão Nile válida.
campi_002,sim,sim,sim,sim,0,0,Expressão Nile válida.
campi_003,sim,sim,sim,sim,0,0,Expressão Nile válida.
campi_004,sim,sim,sim,sim,0,0,Expressão Nile válida.
campi_005,sim,sim,sim,sim,0,0,Expressão Nile válida.
campi_006,sim,sim,sim,sim,0,0,Expressão Nile válida.
campi_007,sim,sim,sim,sim,0,0,Expressão Nile válida.
campi_008,sim,sim,sim,sim,0,0,Expressão Nile válida.
campi_009,sim,sim,sim,sim,0,0,Expressão Nile válida.
campi_010,sim,sim,sim,sim,0,0,Expressão Nile válida.


Bloco 6 concluído
Casos positivos: 50
Casos negativos: 25
PSR de referência: 1.0000
Taxa estrutural de referência: 1.0000
Status: OK


## Bloco 7 - Definição e teste das métricas comuns

### Objetivo

Este bloco implementa o padrão de avaliação utilizado em todas as estratégias da F3. As métricas são centralizadas na F0 para impedir que cada notebook adote regras diferentes de comparação.

### Métricas implementadas

O módulo `nile_metrics.py` calcula:

- **PSR**, indicador de aceitação sintática;
- **EM**, correspondência textual exata entre saída e referência;
- **ED**, distância de edição;
- **NED**, distância de edição normalizada;
- **SLA-S**, aderência estrutural binária;
- **SLA-F**, aderência complementar por campos.

PSR mede apenas a aceitação pelo parser. Uma expressão pode ser sintaticamente válida e ainda divergir da estrutura esperada. SLA-S compara a estrutura de maneira binária, enquanto SLA-F registra a proporção de campos atômicos correspondentes.

### Tratamento estrutural

A comparação considera a natureza dos componentes:

- coleções sem ordem semântica, como targets e conjuntos associados a `allow`, `block`, `set` e `unset`, podem ser comparadas sem depender da ordem textual;
- sequências de middleboxes preservam a ordem quando ela faz parte da estrutura;
- escopos, operadores, valores, unidades e restrições temporais são comparados por campos formais.

### Testes determinísticos

O bloco cria casos para verificar propriedades esperadas, incluindo:

- identidade perfeita;
- diferença apenas textual;
- alteração estrutural;
- saída sintaticamente inválida;
- diferença na ordem de componentes ordenados e não ordenados;
- campos ausentes ou adicionais.

Depois, cada referência é comparada consigo mesma. O resultado esperado é EM, PSR, SLA-S e SLA-F iguais a 1, com ED e NED iguais a 0.

### Artefato produzido

O código é persistido em:

```text
nile_metrics.py
```

### Resultado esperado

Todos os testes devem atender às asserções registradas. O módulo passa então a ser a única implementação oficial das métricas para a F3 e a consolidação posterior.

In [7]:
# ----------------------------------------------------------
# 7.1 Conteúdo do módulo nile_metrics.py
# ----------------------------------------------------------

NILE_METRICS_CODE = r"""from __future__ import annotations

import copy
from collections import Counter
from typing import Any, Dict, List, Tuple


def normalize_nile_text(text: Any) -> str:
    '''Padroniza somente espaços externos aos valores entre aspas simples.'''
    if text is None:
        return ""

    texto = str(text)
    resultado: List[str] = []
    espaco_pendente = False
    dentro_string = False

    for caractere in texto.strip():
        if caractere == "'":
            if not dentro_string and espaco_pendente:
                if resultado and resultado[-1] not in " (":
                    resultado.append(" ")
                espaco_pendente = False

            resultado.append(caractere)
            dentro_string = not dentro_string
            continue

        if dentro_string:
            resultado.append(caractere)
            continue

        if caractere.isspace():
            espaco_pendente = True
            continue

        if caractere == "(":
            while resultado and resultado[-1] == " ":
                resultado.pop()
            resultado.append(caractere)
            espaco_pendente = False
            continue

        if caractere == ")":
            while resultado and resultado[-1] == " ":
                resultado.pop()
            resultado.append(caractere)
            espaco_pendente = False
            continue

        if caractere == ",":
            while resultado and resultado[-1] == " ":
                resultado.pop()
            resultado.append(caractere)
            espaco_pendente = True
            continue

        if espaco_pendente and resultado and resultado[-1] not in " (":
            resultado.append(" ")

        espaco_pendente = False
        resultado.append(caractere)

    return "".join(resultado).strip()


def levenshtein_distance(reference: Any, prediction: Any) -> int:
    a = normalize_nile_text(reference)
    b = normalize_nile_text(prediction)

    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)

    previous = list(range(len(b) + 1))
    for i, char_a in enumerate(a, start=1):
        current = [i]
        for j, char_b in enumerate(b, start=1):
            insertion = current[j - 1] + 1
            deletion = previous[j] + 1
            substitution = previous[j - 1] + (char_a != char_b)
            current.append(min(insertion, deletion, substitution))
        previous = current
    return previous[-1]


def exact_match(reference: Any, prediction: Any) -> int:
    return int(normalize_nile_text(reference) == normalize_nile_text(prediction))


def normalized_edit_distance(reference: Any, prediction: Any) -> float:
    a = normalize_nile_text(reference)
    b = normalize_nile_text(prediction)
    denominator = max(len(a), len(b), 1)
    return levenshtein_distance(a, b) / denominator


def _freeze(value: Any) -> Any:
    if isinstance(value, dict):
        return tuple((key, _freeze(value[key])) for key in sorted(value))
    if isinstance(value, list):
        return tuple(_freeze(item) for item in value)
    return value


def _sort_key(item: Dict[str, Any]) -> Tuple[str, str, str, str]:
    return (
        str(item.get("kind", "")),
        str(item.get("constraint", "")),
        str(item.get("value", "")),
        str(item.get("unit", "")),
    )


def normalized_ast(ast: Dict[str, Any], ignore_intent_id: bool = True) -> Dict[str, Any]:
    '''Normaliza apenas listas cuja ordem não altera a intenção representada.'''
    if not isinstance(ast, dict):
        return {}

    scope_original = ast.get("scope") or {}
    scope = {
        "type": scope_original.get("type"),
        "from": copy.deepcopy(scope_original.get("from")),
        "to": copy.deepcopy(scope_original.get("to")),
        "targets": sorted(
            copy.deepcopy(scope_original.get("targets") or []),
            key=_sort_key,
        ),
    }

    operations = []
    for operation in ast.get("operations") or []:
        if not isinstance(operation, dict):
            continue

        operator = operation.get("operator")
        items = copy.deepcopy(operation.get("items") or [])

        if operator in {"allow", "block", "set", "unset"}:
            items = sorted(items, key=_sort_key)
        # Em add, a ordem dos middleboxes é preservada.

        operations.append({"operator": operator, "items": items})

    result = {
        "scope": scope,
        "operations": operations,
        "interval": copy.deepcopy(ast.get("interval")),
    }

    if not ignore_intent_id:
        result["intent_id"] = ast.get("intent_id")

    return result


def structural_exact(reference_ast: Dict[str, Any], prediction_ast: Dict[str, Any]) -> int:
    if not isinstance(reference_ast, dict) or not isinstance(prediction_ast, dict):
        return 0

    return int(
        _freeze(normalized_ast(reference_ast, ignore_intent_id=True))
        == _freeze(normalized_ast(prediction_ast, ignore_intent_id=True))
    )


def ast_atomic_fields(ast: Dict[str, Any]) -> List[Tuple[str, Any]]:
    '''Extrai campos atômicos e preserva ordem somente quando ela é semântica.'''
    if not isinstance(ast, dict):
        return []

    normalized = normalized_ast(ast, ignore_intent_id=True)
    fields: List[Tuple[str, Any]] = []

    scope = normalized.get("scope") or {}
    fields.append(("scope.type", scope.get("type")))

    source = scope.get("from")
    if isinstance(source, dict):
        fields.append(("scope.source.kind", source.get("kind")))
        fields.append(("scope.source.value", source.get("value")))

    destination = scope.get("to")
    if isinstance(destination, dict):
        fields.append(("scope.destination.kind", destination.get("kind")))
        fields.append(("scope.destination.value", destination.get("value")))

    for target in scope.get("targets") or []:
        if isinstance(target, dict):
            fields.append(("scope.target.kind", target.get("kind")))
            fields.append(("scope.target.value", target.get("value")))

    for op_index, operation in enumerate(normalized.get("operations") or []):
        if not isinstance(operation, dict):
            continue

        operator = operation.get("operator")
        op_prefix = f"operations[{op_index}]"
        fields.append((f"{op_prefix}.operator", operator))

        for item_index, item in enumerate(operation.get("items") or []):
            if not isinstance(item, dict):
                continue

            kind = item.get("kind")

            if operator == "add":
                # A posição integra o caminho porque add pode representar uma cadeia.
                item_prefix = f"{op_prefix}.items[{item_index}]"
                fields.append((f"{item_prefix}.kind", kind))
                fields.append((f"{item_prefix}.value", item.get("value")))

            elif operator in {"allow", "block"}:
                # Sem índice: a ordem dos itens de ACL não altera a intenção.
                fields.append((f"{op_prefix}.items.kind", kind))
                fields.append((f"{op_prefix}.items.value", item.get("value")))

            elif operator == "set":
                # Políticas são avaliadas por campos atômicos e por tipo.
                policy_prefix = f"{op_prefix}.policies.{kind}"
                fields.append((f"{op_prefix}.policies.kind", kind))
                fields.append((f"{policy_prefix}.constraint", item.get("constraint")))
                fields.append((f"{policy_prefix}.value", item.get("value")))
                fields.append((f"{policy_prefix}.unit", item.get("unit")))

            elif operator == "unset":
                fields.append((f"{op_prefix}.policies.kind", kind))

    interval = normalized.get("interval")
    if isinstance(interval, dict):
        for name in ("start", "end"):
            ref = interval.get(name)
            if isinstance(ref, dict):
                fields.append((f"interval.{name}.kind", ref.get("kind")))
                fields.append((f"interval.{name}.value", ref.get("value")))

    return fields


def field_level_adherence(reference_ast: Dict[str, Any], prediction_ast: Dict[str, Any]) -> float:
    reference_fields = Counter(ast_atomic_fields(reference_ast))
    prediction_fields = Counter(ast_atomic_fields(prediction_ast))

    reference_total = sum(reference_fields.values())
    prediction_total = sum(prediction_fields.values())

    if reference_total == 0 and prediction_total == 0:
        return 1.0
    if reference_total == 0 or prediction_total == 0:
        return 0.0

    matches = sum((reference_fields & prediction_fields).values())
    denominator = max(reference_total, prediction_total)
    return matches / denominator if denominator else 1.0


def evaluate_pair(
    reference_nile: Any,
    prediction_nile: Any,
    validator,
) -> Dict[str, Any]:
    reference_text = normalize_nile_text(reference_nile)
    prediction_text = normalize_nile_text(prediction_nile)

    reference_validation = validator.validate(reference_text)
    prediction_validation = validator.validate(prediction_text)

    reference_ast = reference_validation.get("ast") if reference_validation.get("valid") else None
    prediction_ast = prediction_validation.get("ast") if prediction_validation.get("valid") else None

    return {
        "psr": int(prediction_validation.get("syntax_valid", False)),
        "structural_valid": int(prediction_validation.get("structural_valid", False)),
        "em": exact_match(reference_text, prediction_text),
        "ed": levenshtein_distance(reference_text, prediction_text),
        "ned": normalized_edit_distance(reference_text, prediction_text),
        "sla_s": structural_exact(reference_ast, prediction_ast),
        "sla_f": field_level_adherence(reference_ast, prediction_ast),
        "reference_valid": bool(reference_validation.get("valid", False)),
        "prediction_valid": bool(prediction_validation.get("valid", False)),
        "prediction_feedback": prediction_validation.get("feedback", ""),
    }
"""


# ----------------------------------------------------------
# 7.2 Salvamento e compilação do módulo
# ----------------------------------------------------------

NILE_METRICS_PATH.write_text(NILE_METRICS_CODE.strip() + "\n", encoding="utf-8")
py_compile.compile(str(NILE_METRICS_PATH), doraise=True)


# ----------------------------------------------------------
# 7.3 Importação controlada das métricas
# ----------------------------------------------------------

importlib.invalidate_caches()

if "nile_metrics" in sys.modules:
    del sys.modules["nile_metrics"]

from nile_metrics import evaluate_pair, normalize_nile_text


# ----------------------------------------------------------
# 7.4 Definição dos testes determinísticos das métricas
# ----------------------------------------------------------

referencia_acl = (
    "define intent uniIntent: from endpoint('internet') to endpoint('network') "
    "block protocol('ftp')"
)

referencia_acl_multipla = (
    "define intent uniIntent: for endpoint('network') "
    "allow protocol('HTTP'), protocol('HTTPS')"
)

referencia_middleboxes = (
    "define intent uniIntent: for endpoint('network') "
    "add middlebox('firewall'), middlebox('dpi')"
)

referencia_quota = (
    "define intent uniIntent: for group('students') "
    "set quota('download', '100', 'gb/wk')"
)

referencia_espaco_literal = (
    "define intent uniIntent: for endpoint('A  B') block protocol('ftp')"
)

casos_metricas = [
    {
        "case": "identica",
        "reference": referencia_acl,
        "prediction": referencia_acl,
    },
    {
        "case": "valor_alterado",
        "reference": referencia_acl,
        "prediction": referencia_acl.replace("'ftp'", "'http'"),
    },
    {
        "case": "rota_invertida",
        "reference": referencia_acl,
        "prediction": (
            "define intent uniIntent: from endpoint('network') to endpoint('internet') "
            "block protocol('ftp')"
        ),
    },
    {
        "case": "ordem_acl_alterada",
        "reference": referencia_acl_multipla,
        "prediction": (
            "define intent uniIntent: for endpoint('network') "
            "allow protocol('HTTPS'), protocol('HTTP')"
        ),
    },
    {
        "case": "ordem_middlebox_alterada",
        "reference": referencia_middleboxes,
        "prediction": (
            "define intent uniIntent: for endpoint('network') "
            "add middlebox('dpi'), middlebox('firewall')"
        ),
    },
    {
        "case": "constraint_quota_alterada",
        "reference": referencia_quota,
        "prediction": referencia_quota.replace("'download'", "'upload'"),
    },
    {
        "case": "espaco_interno_alterado",
        "reference": referencia_espaco_literal,
        "prediction": referencia_espaco_literal.replace("'A  B'", "'A B'"),
    },
    {
        "case": "sintaxe_invalida",
        "reference": referencia_acl,
        "prediction": "define intent uniIntent: block protocol('ftp')",
    },
]

resultados_metricas = []
for caso in casos_metricas:
    metricas = evaluate_pair(caso["reference"], caso["prediction"], validator)
    resultados_metricas.append({"case": caso["case"], **metricas})


df_testes_metricas = pd.DataFrame(resultados_metricas)


# ----------------------------------------------------------
# 7.5 Asserções das propriedades esperadas
# ----------------------------------------------------------

metricas_por_caso = df_testes_metricas.set_index("case")

identica = metricas_por_caso.loc["identica"]
valor_alterado = metricas_por_caso.loc["valor_alterado"]
rota_invertida = metricas_por_caso.loc["rota_invertida"]
ordem_acl = metricas_por_caso.loc["ordem_acl_alterada"]
ordem_middlebox = metricas_por_caso.loc["ordem_middlebox_alterada"]
constraint_quota = metricas_por_caso.loc["constraint_quota_alterada"]
espaco_interno = metricas_por_caso.loc["espaco_interno_alterado"]
invalida = metricas_por_caso.loc["sintaxe_invalida"]

assert identica["psr"] == 1
assert identica["em"] == 1
assert identica["ed"] == 0
assert identica["ned"] == 0
assert identica["sla_s"] == 1
assert identica["sla_f"] == 1

assert valor_alterado["psr"] == 1
assert valor_alterado["em"] == 0
assert valor_alterado["sla_s"] == 0
assert 0 < valor_alterado["sla_f"] < 1

assert rota_invertida["psr"] == 1
assert rota_invertida["sla_s"] == 0
assert 0 < rota_invertida["sla_f"] < 1

assert ordem_acl["em"] == 0
assert ordem_acl["sla_s"] == 1
assert ordem_acl["sla_f"] == 1

assert ordem_middlebox["sla_s"] == 0
assert 0 < ordem_middlebox["sla_f"] < 1

assert constraint_quota["sla_s"] == 0
assert abs(float(constraint_quota["sla_f"]) - 0.875) < 1e-12

assert espaco_interno["em"] == 0
assert espaco_interno["ned"] > 0
assert espaco_interno["sla_s"] == 0
assert 0 < espaco_interno["sla_f"] < 1
assert "'A  B'" in normalize_nile_text(referencia_espaco_literal)

assert invalida["psr"] == 0
assert invalida["sla_s"] == 0
assert invalida["sla_f"] == 0


# ----------------------------------------------------------
# 7.6 Autocomparação das 50 referências
# ----------------------------------------------------------

resultados_autocomparacao = []
for linha in df_campi.itertuples(index=False):
    metricas = evaluate_pair(linha.nile_canonical, linha.nile_canonical, validator)
    resultados_autocomparacao.append({"id": linha.id, **metricas})


df_autocomparacao = pd.DataFrame(resultados_autocomparacao)

colunas_perfeitas = ["psr", "structural_valid", "em", "sla_s", "sla_f"]
if not (df_autocomparacao[colunas_perfeitas] == 1).all().all():
    raise AssertionError("A autocomparação das referências não produziu métricas perfeitas.")

if not (df_autocomparacao["ed"] == 0).all():
    raise AssertionError("A ED da autocomparação deveria ser zero.")

if not (df_autocomparacao["ned"] == 0).all():
    raise AssertionError("A NED da autocomparação deveria ser zero.")


# ----------------------------------------------------------
# 7.7 Tabela dos testes das métricas
# ----------------------------------------------------------

exibir_tabela(
    df_testes_metricas[[
        "case",
        "psr",
        "structural_valid",
        "em",
        "ed",
        "ned",
        "sla_s",
        "sla_f",
        "prediction_valid",
    ]],
    "Testes determinísticos das métricas comuns",
    altura_px=420,
)


# ----------------------------------------------------------
# 7.8 Saída do bloco
# ----------------------------------------------------------

print("Bloco 7 concluído")
print(f"Módulo gerado: {NILE_METRICS_PATH}")
print(f"Casos determinísticos de métricas: {len(df_testes_metricas)}")
print(f"Referências autocomparadas: {len(df_autocomparacao)}")
print("Status: OK")

caso,PSR,estrutura válida,EM,ED,NED,SLA-S,SLA-F,saída válida
idêntica,1,1,1,0,0.000000,1,1.000000,sim
valor alterado,1,1,0,2,0.020833,0,0.875000,sim
rota invertida,1,1,0,14,0.147368,0,0.750000,sim
ordem dos itens de ACL alterada,1,1,0,2,0.022222,1,1.000000,sim
ordem dos middleboxes alterada,1,1,0,16,0.173913,0,0.750000,sim
constraint de quota alterada,1,1,0,4,0.047619,0,0.875000,sim
espaço interno alterado,1,1,0,1,0.014925,0,0.833333,sim
sintaxe inválida,0,0,0,49,0.515789,0,0.000000,não


Bloco 7 concluído
Módulo gerado: /kaggle/working/f0_operacional/nile_metrics.py
Casos determinísticos de métricas: 8
Referências autocomparadas: 50
Status: OK


## Bloco 8 - Definição dos cinco folds experimentais

### Objetivo

Este bloco cria as divisões utilizadas para avaliar o modelo aluno. O desenho garante que todos os exemplos sejam testados uma vez e que as estratégias utilizem exatamente os mesmos conjuntos de treino e teste.

### Configuração

São definidos:

- cinco folds;
- semente global `42`;
- 50 exemplos no total;
- 10 exemplos de teste por fold;
- 40 exemplos de treino por fold.

### Estratificação

A construção considera classes estruturais derivadas das referências, de modo a distribuir, tanto quanto possível, diferentes formas de escopo e operação entre os folds. O procedimento é determinístico e depende apenas dos atributos congelados na F0.

### Verificações de integridade

O bloco confirma que:

- cada fold possui 50 linhas de associação entre ID, fold e split;
- cada fold contém exatamente 40 registros de treino e 10 de teste;
- os conjuntos de treino e teste são disjuntos;
- cada ID aparece como teste em exatamente um fold;
- a união dos testes cobre os 50 exemplos;
- não existem IDs ausentes ou duplicados;
- a distribuição estrutural pode ser auditada por fold.

### Artefato produzido

As divisões são salvas em:

```text
folds.csv
```

O arquivo registra, para cada ID e fold, se o exemplo pertence ao treino ou ao teste.

### Tabelas produzidas

O bloco apresenta:

- contagem de treino e teste por fold;
- distribuição das classes estruturais nos testes;
- lista dos IDs de teste de cada fold.

### Resultado esperado

Os cinco folds devem estar completos, sem sobreposição entre treino e teste e sem deixar exemplos fora da avaliação.

In [8]:
# ----------------------------------------------------------
# 8.1 Verificação das classes utilizadas na estratificação
# ----------------------------------------------------------

contagem_familias = df_campi["primary_family"].value_counts().sort_index()

if (contagem_familias < N_FOLDS).any():
    classes_invalidas = contagem_familias[contagem_familias < N_FOLDS]
    raise ValueError(
        "Não é possível aplicar StratifiedKFold com as classes atuais:\n"
        + classes_invalidas.to_string()
    )


# ----------------------------------------------------------
# 8.2 Construção determinística dos folds
# ----------------------------------------------------------

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED,
)

registros_folds = []

for fold, (indices_treino, indices_teste) in enumerate(
    skf.split(df_campi, df_campi["primary_family"]),
    start=1,
):
    conjunto_teste = set(indices_teste.tolist())

    for indice_linha, linha in df_campi.reset_index(drop=True).iterrows():
        split = "test" if indice_linha in conjunto_teste else "train"

        registros_folds.append({
            "fold": fold,
            "id": linha["id"],
            "split": split,
            "primary_family": linha["primary_family"],
            "has_temporal": bool(linha["has_temporal"]),
            "has_route": bool(linha["has_route"]),
            "n_operations": int(linha["n_operations"]),
            "n_items": int(linha["n_items"]),
            "complexity_score": int(linha["complexity_score"]),
            "seed": SEED,
        })


df_folds = pd.DataFrame(registros_folds)


# ----------------------------------------------------------
# 8.3 Verificações de integridade dos folds
# ----------------------------------------------------------

contagens_split = (
    df_folds
    .groupby(["fold", "split"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for linha in contagens_split.itertuples(index=False):
    if linha.test != EXPECTED_TEST_PER_FOLD:
        raise AssertionError(
            f"O fold {linha.fold} possui {linha.test} testes; esperado: {EXPECTED_TEST_PER_FOLD}."
        )
    if linha.train != EXPECTED_EXAMPLES - EXPECTED_TEST_PER_FOLD:
        raise AssertionError(
            f"O fold {linha.fold} possui {linha.train} treinos; esperado: 40."
        )

ocorrencias_teste = (
    df_folds.loc[df_folds["split"] == "test"]
    .groupby("id")
    .size()
)

if len(ocorrencias_teste) != EXPECTED_EXAMPLES:
    raise AssertionError("Nem todos os exemplos aparecem no conjunto de teste.")

if not (ocorrencias_teste == 1).all():
    raise AssertionError("Algum exemplo aparece mais de uma vez como teste.")


# ----------------------------------------------------------
# 8.4 Salvamento dos folds
# ----------------------------------------------------------

df_folds.to_csv(FOLDS_PATH, index=False, encoding="utf-8")


# ----------------------------------------------------------
# 8.5 Tabela de contagens por fold
# ----------------------------------------------------------

exibir_tabela(
    contagens_split,
    "Quantidade de exemplos de treino e teste por fold",
    altura_px=260,
)


# ----------------------------------------------------------
# 8.6 Tabela da distribuição estrutural nos testes
# ----------------------------------------------------------

distribuicao_familias_teste = (
    df_folds.loc[df_folds["split"] == "test"]
    .groupby(["fold", "primary_family"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

exibir_tabela(
    distribuicao_familias_teste,
    "Distribuição das famílias estruturais nos conjuntos de teste",
    altura_px=300,
)


# ----------------------------------------------------------
# 8.7 Tabela dos identificadores de teste
# ----------------------------------------------------------

ids_teste_por_fold = (
    df_folds.loc[df_folds["split"] == "test"]
    .sort_values(["fold", "id"])
    .groupby("fold")["id"]
    .apply(lambda valores: ", ".join(valores))
    .reset_index(name="test_ids")
)

exibir_tabela(
    ids_teste_por_fold,
    "Identificadores de teste em cada fold",
    altura_px=300,
)


# ----------------------------------------------------------
# 8.8 Saída do bloco
# ----------------------------------------------------------

print("Bloco 8 concluído")
print(f"Folds: {N_FOLDS}")
print(f"Testes por fold: {EXPECTED_TEST_PER_FOLD}")
print(f"Exemplos únicos em teste: {len(ocorrencias_teste)}")
print("Status: OK")

split,fold,teste,treino
,1,10,40
,2,10,40
,3,10,40
,4,10,40
,5,10,40


primary_family,fold,acl,middlebox,mixed,qos
,1,1,2,4,3
,2,1,2,4,3
,3,2,2,3,3
,4,2,1,3,4
,5,2,1,4,3


fold,IDs de teste
1,"campi_001, campi_002, campi_011, campi_012, campi_020, campi_023, campi_024, campi_027, campi_039, campi_044"
2,"campi_006, campi_009, campi_010, campi_017, campi_028, campi_031, campi_034, campi_041, campi_043, campi_048"
3,"campi_003, campi_007, campi_014, campi_018, campi_025, campi_026, campi_033, campi_035, campi_045, campi_047"
4,"campi_005, campi_016, campi_019, campi_021, campi_032, campi_036, campi_037, campi_040, campi_046, campi_050"
5,"campi_004, campi_008, campi_013, campi_015, campi_022, campi_029, campi_030, campi_038, campi_042, campi_049"


Bloco 8 concluído
Folds: 5
Testes por fold: 10
Exemplos únicos em teste: 50
Status: OK


## Bloco 9 - Biblioteca dos 11 prompts

### Objetivo

Este bloco registra os templates utilizados pela F2 e pela F3. A biblioteca centralizada assegura que as solicitações ao modelo professor e ao modelo aluno tenham versões congeladas, placeholders conhecidos e hashes rastreáveis.

### Organização dos templates

São preparados onze templates:

1. geração de R1;
2. geração de R2;
3. geração de R3;
4. revisão de justificativa reprovada;
5. F3-A, tradução direta para Nile;
6. F3-C, geração da IR;
7. F3-C, geração de Nile a partir da IR;
8. F3-D, tradução com R3;
9. F3-E, geração da IR com R1;
10. F3-E, geração de Nile com R2;
11. autocorreção da saída Nile.

F3-B reutiliza a saída e os prompts da tradução direta e aplica o template de autocorreção.

### Regras comuns

O bloco separa:

- instruções destinadas ao modelo professor;
- instruções destinadas ao modelo aluno;
- política de idioma;
- preservação de campos e valores formais;
- formato de saída JSON;
- formato de saída Nile;
- proibição de explicações adicionais quando o template exige somente o artefato.

Os prompts do professor podem ser documentados em português, mas as justificativas R1, R2 e R3 são solicitadas em inglês. Campos da IR, operadores Nile, identificadores, unidades e valores literais permanecem inalterados.

### Placeholders

Cada template possui um conjunto explícito de placeholders. O bloco verifica:

- presença dos campos esperados;
- ausência de placeholders desconhecidos;
- numeração e identificadores oficiais;
- consistência entre biblioteca, arquivos individuais e índice.

### Artefatos produzidos

São salvos:

- `biblioteca_prompts.json`;
- onze arquivos em `prompts/`.

### Resultado esperado

A tabela final apresenta o ID, a finalidade, o arquivo, os placeholders e o hash de cada template. Essa biblioteca passa a ser consumida sem alterações pelas fases posteriores.

In [9]:
# ----------------------------------------------------------
# 9.1 Função auxiliar para registrar os Templates de Prompt
# ----------------------------------------------------------

prompts = {}

def registrar_prompt(
    tp,
    prompt_id,
    arquivo,
    fase,
    modelo,
    estrategia,
    nome_tabela,
    objetivo,
    placeholders,
    texto,
):
    prompts[prompt_id] = {
        "tp": tp,
        "prompt_id": prompt_id,
        "arquivo": arquivo,
        "fase": fase,
        "modelo": modelo,
        "estrategia": estrategia,
        "nome_tabela": nome_tabela,
        "objetivo": objetivo,
        "placeholders": list(placeholders),
        "texto": texto.strip(),
    }


# ----------------------------------------------------------
# 9.2 Abertura padrão para o modelo professor
# ----------------------------------------------------------

abertura_modelo_professor = """
Este comando faz parte de um fluxo reprodutível de tradução de entradas em linguagem natural para a linguagem formal Nile.

Definições usadas neste comando:
- Linguagem natural (NL): frase de entrada escrita em inglês.
- Nile: linguagem formal de destino usada como saída esperada do experimento.
- CAMPI: conjunto de referência com 50 pares de entrada em linguagem natural e expressão correspondente em Nile.
- Representação Intermediária (IR): estrutura JSON controlada, derivada deterministicamente da estrutura formal da expressão Nile de referência.
- R1: justificativa explícita para a passagem da linguagem natural para a Representação Intermediária.
- R2: justificativa explícita para a passagem da Representação Intermediária para Nile.
- R3: justificativa explícita para a passagem direta da linguagem natural para Nile.

O fluxo utiliza dois papéis:
- modelo professor: prepara R1, R2 e R3 na F2;
- modelo aluno: executa as estratégias experimentais na F3.

Neste comando, sua função é atuar como modelo professor. Produza somente o artefato solicitado, no formato especificado.
""".strip()


# ----------------------------------------------------------
# 9.3 Abertura padrão para o modelo aluno
# ----------------------------------------------------------

abertura_modelo_aluno = """
Este comando faz parte de um fluxo reprodutível de tradução de entradas em linguagem natural para a linguagem formal Nile.

Definições usadas neste comando:
- Linguagem natural (NL): frase de entrada escrita em inglês.
- Nile: linguagem formal de destino usada como saída final do experimento.
- Representação Intermediária (IR): estrutura JSON controlada que organiza os componentes formais da intenção.
- R1: justificativa explícita para a passagem da linguagem natural para a Representação Intermediária.
- R2: justificativa explícita para a passagem da Representação Intermediária para Nile.
- R3: justificativa explícita para a passagem direta da linguagem natural para Nile.

Neste comando, sua função é atuar como modelo aluno.
""".strip()


# ----------------------------------------------------------
# 9.4 Regras comuns de idioma e preservação
# ----------------------------------------------------------

regras_idioma_e_preservacao = """
Regras comuns de idioma e preservação:
1. Use português do Brasil em justificativas e textos explicativos solicitados.
2. Mantenha em inglês nomes de chaves JSON, nomes de campos estruturais, valores controlados, tokens da linguagem Nile, identificadores, nomes de serviços, protocolos, unidades e strings extraídas do CAMPI.
3. Não traduza entradas em linguagem natural, expressões Nile, identificadores, nomes de protocolos, nomes de serviços, nomes de grupos, nomes de endpoints ou valores fornecidos.
4. Não acrescente informações que não estejam sustentadas pelos dados apresentados.
""".strip()


# ----------------------------------------------------------
# 9.5 Regras comuns para saídas JSON
# ----------------------------------------------------------

regras_json_valido = """
Regras comuns para saídas em JSON:
1. Responda apenas com JSON válido.
2. Não use Markdown.
3. Não use bloco de código.
4. Não escreva comentários externos.
5. Use aspas duplas em todas as chaves e strings.
6. Não use vírgula após o último item de objetos ou listas.
7. Use true, false e null quando necessário, conforme o padrão JSON.
8. Não inclua campos que não tenham sido solicitados.
""".strip()


regras_modelo_professor_json = f"""
{regras_idioma_e_preservacao}

{regras_json_valido}
""".strip()


# ----------------------------------------------------------
# 9.6 Regras comuns para saídas do modelo aluno
# ----------------------------------------------------------

regras_saida_modelo_aluno = """
Regras comuns para saídas do modelo aluno:
1. Produza apenas o artefato solicitado.
2. Não escreva explicações.
3. Não escreva comentários.
4. Não use Markdown.
5. Não use bloco de código.
6. Não repita a entrada de teste.
7. Não use a saída de referência do exemplo de teste.
8. Preserve a intenção da entrada em linguagem natural.
9. Mantenha em inglês os termos, nomes e valores que fazem parte da entrada, da Representação Intermediária, dos exemplos demonstrativos ou da expressão Nile.
""".strip()


# ----------------------------------------------------------
# 9.7 TP1 - Geração da justificativa R1
# ----------------------------------------------------------

registrar_prompt(
    tp="TP1",
    prompt_id="f2_geracao_r1",
    arquivo="f2_geracao_r1.txt",
    fase="F2",
    modelo="DeepSeek-V4",
    estrategia="preparacao_r1",
    nome_tabela="Geração da justificativa explícita R1",
    objetivo=(
        "Gerar R1 para explicar a passagem da linguagem natural "
        "para a Representação Intermediária."
    ),
    placeholders=["IR_JSON"],
    texto=f"""
{abertura_modelo_professor}

{regras_modelo_professor_json}

Tarefa:
Gere uma justificativa R1 para cada exemplo fornecido.

A justificativa R1 explica a passagem da entrada em linguagem natural para a Representação Intermediária. Ela deve explicitar quais elementos da frase foram identificados e como eles foram organizados na estrutura intermediária.

Exemplos fornecidos:
{{{{IR_JSON}}}}

Requisitos:
1. Gere exatamente uma justificativa R1 para cada exemplo fornecido.
2. Preserve o identificador original.
3. A justificativa deve ser curta, objetiva e verificável.
4. A justificativa deve mencionar os elementos estruturais efetivamente presentes na Representação Intermediária.
5. Não inclua informação que não esteja sustentada pela entrada ou pela IR.
6. A justificativa deve ser escrita em português do Brasil.
7. Não altere a entrada em linguagem natural, a Representação Intermediária ou a expressão Nile de referência.
8. Preserve em inglês termos técnicos, identificadores, serviços, protocolos, grupos, endpoints e demais valores dos dados.

Formato obrigatório:
{{
  "items": [
    {{
      "id": "campi_001",
      "r1": "Justificativa em português do Brasil."
    }}
  ]
}}
""",
)


# ----------------------------------------------------------
# 9.8 TP2 - Geração da justificativa R2
# ----------------------------------------------------------

registrar_prompt(
    tp="TP2",
    prompt_id="f2_geracao_r2",
    arquivo="f2_geracao_r2.txt",
    fase="F2",
    modelo="DeepSeek-V4",
    estrategia="preparacao_r2",
    nome_tabela="Geração da justificativa explícita R2",
    objetivo=(
        "Gerar R2 para explicar a passagem da Representação "
        "Intermediária para Nile."
    ),
    placeholders=["IR_JSON"],
    texto=f"""
{abertura_modelo_professor}

{regras_modelo_professor_json}

Tarefa:
Gere uma justificativa R2 para cada exemplo fornecido.

A justificativa R2 explica como os elementos da Representação Intermediária são compostos para formar a expressão correta em Nile.

Exemplos fornecidos:
{{{{IR_JSON}}}}

Requisitos:
1. Gere exatamente uma justificativa R2 para cada exemplo fornecido.
2. Preserve o identificador original.
3. A justificativa deve ser curta, objetiva e verificável.
4. A justificativa deve relacionar os campos da IR à composição da expressão Nile.
5. Mencione escopo, operações, argumentos e restrições temporais somente quando estiverem presentes.
6. Não inclua informação que não esteja sustentada pela IR ou pela expressão Nile.
7. A justificativa deve ser escrita em português do Brasil.
8. Não altere a entrada em linguagem natural, a Representação Intermediária ou a expressão Nile de referência.
9. Preserve em inglês termos técnicos, identificadores, serviços, protocolos, grupos, endpoints e demais valores dos dados.

Formato obrigatório:
{{
  "items": [
    {{
      "id": "campi_001",
      "r2": "Justificativa em português do Brasil."
    }}
  ]
}}
""",
)


# ----------------------------------------------------------
# 9.9 TP3 - Geração da justificativa R3
# ----------------------------------------------------------

registrar_prompt(
    tp="TP3",
    prompt_id="f2_geracao_r3",
    arquivo="f2_geracao_r3.txt",
    fase="F2",
    modelo="DeepSeek-V4",
    estrategia="preparacao_r3",
    nome_tabela="Geração da justificativa explícita R3",
    objetivo=(
        "Gerar R3 para explicar a passagem direta da linguagem "
        "natural para Nile."
    ),
    placeholders=["CAMPI_JSON"],
    texto=f"""
{abertura_modelo_professor}

{regras_modelo_professor_json}

Tarefa:
Gere uma justificativa R3 para cada exemplo fornecido.

A justificativa R3 explica a tradução direta da entrada em linguagem natural para a expressão correta em Nile. Ela será usada somente como artefato demonstrativo em exemplos few-shot.

Exemplos fornecidos:
{{{{CAMPI_JSON}}}}

Requisitos:
1. Gere exatamente uma justificativa R3 para cada exemplo fornecido.
2. Preserve o identificador original.
3. A justificativa deve ser curta, objetiva e verificável.
4. A justificativa deve mencionar ação, escopo, entidades, relações, argumentos e restrições temporais somente quando estiverem presentes.
5. Não inclua informação que não esteja sustentada pelo par NL-Nile.
6. A justificativa deve ser escrita em português do Brasil.
7. Não altere a entrada em linguagem natural ou a expressão Nile de referência.
8. Preserve em inglês termos técnicos, identificadores, serviços, protocolos, grupos, endpoints e demais valores dos dados.

Formato obrigatório:
{{
  "items": [
    {{
      "id": "campi_001",
      "r3": "Justificativa em português do Brasil."
    }}
  ]
}}
""",
)


# ----------------------------------------------------------
# 9.10 TP4 - Revisão de justificativa reprovada
# ----------------------------------------------------------

registrar_prompt(
    tp="TP4",
    prompt_id="f2_revisao_justificativa_reprovada",
    arquivo="f2_revisao_justificativa_reprovada.txt",
    fase="F2",
    modelo="DeepSeek-V4",
    estrategia="revisao_r1_r2_r3",
    nome_tabela="Revisão de justificativa R1, R2 ou R3",
    objetivo="Corrigir uma justificativa reprovada pela validação da F2.",
    placeholders=[
        "TIPO_JUSTIFICATIVA",
        "EXEMPLO_JSON",
        "JUSTIFICATIVA_REPROVADA",
        "ERROS_VALIDACAO_JSON",
    ],
    texto=f"""
{abertura_modelo_professor}

{regras_modelo_professor_json}

Tarefa:
Corrija a justificativa reprovada.

Tipo de justificativa:
{{{{TIPO_JUSTIFICATIVA}}}}

Exemplo:
{{{{EXEMPLO_JSON}}}}

Justificativa reprovada:
{{{{JUSTIFICATIVA_REPROVADA}}}}

Erros de validação:
{{{{ERROS_VALIDACAO_JSON}}}}

Requisitos:
1. Preserve o identificador do exemplo.
2. Preserve o tipo da justificativa.
3. Corrija somente os problemas informados.
4. Produza uma justificativa curta, objetiva e verificável.
5. Remova afirmações não sustentadas pelos dados.
6. A justificativa deve ser escrita em português do Brasil.
7. Não altere a entrada em linguagem natural, a Representação Intermediária ou a expressão Nile.
8. Preserve em inglês termos técnicos, identificadores, serviços, protocolos, grupos, endpoints e demais valores dos dados.

Formato obrigatório:
{{
  "id": "campi_001",
  "tipo": "R1",
  "justificativa": "Justificativa corrigida em português do Brasil."
}}
""",
)


# ----------------------------------------------------------
# 9.11 TP5 - Estratégia A: tradução direta para Nile
# ----------------------------------------------------------

registrar_prompt(
    tp="TP5",
    prompt_id="f3a_traducao_direta",
    arquivo="f3a_traducao_direta.txt",
    fase="F3-A",
    modelo="Qwen2.5-1.5B-Instruct",
    estrategia="A",
    nome_tabela="Tradução direta para Nile",
    objetivo="Traduzir diretamente uma entrada em linguagem natural para Nile.",
    placeholders=["EXEMPLOS_FEW_SHOT", "NL_TESTE"],
    texto=f"""
{abertura_modelo_aluno}

{regras_saida_modelo_aluno}

Tarefa:
Traduza a entrada em linguagem natural escrita em inglês para uma expressão completa em Nile.

Exemplos few-shot:
{{{{EXEMPLOS_FEW_SHOT}}}}

Entrada de teste em linguagem natural:
{{{{NL_TESTE}}}}

Regras específicas:
1. Responda apenas com a expressão final em Nile.
2. A saída deve ser completa, válida e analisável pela gramática Nile.
3. Não invente elementos que não estejam indicados pela entrada.
4. Não gere Representação Intermediária.
5. Não gere justificativa.

Saída Nile:
""",
)


# ----------------------------------------------------------
# 9.12 TP6 - Estratégia C: geração da IR
# ----------------------------------------------------------

registrar_prompt(
    tp="TP6",
    prompt_id="f3c_gerar_ir_teste",
    arquivo="f3c_gerar_ir_teste.txt",
    fase="F3-C",
    modelo="Qwen2.5-1.5B-Instruct",
    estrategia="C",
    nome_tabela="Geração da Representação Intermediária do teste",
    objetivo="Gerar uma IR para a entrada de teste sem usar a expressão Nile de referência.",
    placeholders=["IR_SCHEMA_JSON", "EXEMPLOS_FEW_SHOT_IR", "NL_TESTE"],
    texto=f"""
{abertura_modelo_aluno}

{regras_saida_modelo_aluno}

Tarefa:
Converta a entrada em linguagem natural escrita em inglês para uma Representação Intermediária.

Esquema da Representação Intermediária:
{{{{IR_SCHEMA_JSON}}}}

Exemplos few-shot:
{{{{EXEMPLOS_FEW_SHOT_IR}}}}

Entrada de teste em linguagem natural:
{{{{NL_TESTE}}}}

Regras específicas:
1. Responda apenas com um objeto JSON válido da Representação Intermediária.
2. Use aspas duplas em todas as chaves e strings.
3. Não use vírgula após o último item de objetos ou listas.
4. Não gere a expressão Nile nesta etapa.
5. Não gere justificativa R1 para o exemplo de teste.
6. Preserve a intenção da entrada em linguagem natural.
7. Preserve em inglês termos, nomes, valores e entidades extraídos da entrada de teste.

Representação Intermediária:
""",
)


# ----------------------------------------------------------
# 9.13 TP7 - Estratégia C: geração de Nile a partir da IR
# ----------------------------------------------------------

registrar_prompt(
    tp="TP7",
    prompt_id="f3c_gerar_nile_a_partir_ir",
    arquivo="f3c_gerar_nile_a_partir_ir.txt",
    fase="F3-C",
    modelo="Qwen2.5-1.5B-Instruct",
    estrategia="C",
    nome_tabela="Geração de Nile a partir da Representação Intermediária",
    objetivo="Gerar Nile a partir da IR produzida para o exemplo de teste.",
    placeholders=["EXEMPLOS_FEW_SHOT_IR_NILE", "IR_TESTE"],
    texto=f"""
{abertura_modelo_aluno}

{regras_saida_modelo_aluno}

Tarefa:
Converta a Representação Intermediária abaixo em uma expressão completa em Nile.

Exemplos few-shot:
{{{{EXEMPLOS_FEW_SHOT_IR_NILE}}}}

Representação Intermediária do teste:
{{{{IR_TESTE}}}}

Regras específicas:
1. Responda apenas com a expressão final em Nile.
2. A saída deve ser completa, válida e analisável pela gramática Nile.
3. Use somente as informações da Representação Intermediária do teste.
4. Não repita a Representação Intermediária.
5. Não gere justificativa.

Saída Nile:
""",
)


# ----------------------------------------------------------
# 9.14 TP8 - Estratégia D: tradução com R3
# ----------------------------------------------------------

registrar_prompt(
    tp="TP8",
    prompt_id="f3d_traducao_com_r3",
    arquivo="f3d_traducao_com_r3.txt",
    fase="F3-D",
    modelo="Qwen2.5-1.5B-Instruct",
    estrategia="D",
    nome_tabela="Tradução direta com demonstrações enriquecidas com R3",
    objetivo="Traduzir linguagem natural para Nile usando demonstrações com R3.",
    placeholders=["EXEMPLOS_FEW_SHOT_R3", "NL_TESTE"],
    texto=f"""
{abertura_modelo_aluno}

{regras_saida_modelo_aluno}

Tarefa:
Traduza a entrada em linguagem natural escrita em inglês para uma expressão completa em Nile utilizando os exemplos demonstrativos enriquecidos com R3.

Exemplos few-shot:
{{{{EXEMPLOS_FEW_SHOT_R3}}}}

Entrada de teste em linguagem natural:
{{{{NL_TESTE}}}}

Regras específicas:
1. Responda apenas com a expressão final em Nile.
2. A saída deve ser completa, válida e analisável pela gramática Nile.
3. Use as demonstrações enriquecidas com R3 somente como padrão de tradução.
4. Preserve a intenção da entrada em linguagem natural.
5. Não gere R3 para o exemplo de teste.
6. Não gere Representação Intermediária.

Saída Nile:
""",
)


# ----------------------------------------------------------
# 9.15 TP9 - Estratégia E: geração da IR com R1
# ----------------------------------------------------------

registrar_prompt(
    tp="TP9",
    prompt_id="f3e_gerar_ir_com_r1",
    arquivo="f3e_gerar_ir_com_r1.txt",
    fase="F3-E",
    modelo="Qwen2.5-1.5B-Instruct",
    estrategia="E",
    nome_tabela="Geração da Representação Intermediária com R1",
    objetivo="Gerar uma IR usando demonstrações enriquecidas com R1.",
    placeholders=["IR_SCHEMA_JSON", "EXEMPLOS_FEW_SHOT_R1", "NL_TESTE"],
    texto=f"""
{abertura_modelo_aluno}

{regras_saida_modelo_aluno}

Tarefa:
Converta a entrada em linguagem natural escrita em inglês para uma Representação Intermediária utilizando os exemplos demonstrativos enriquecidos com R1.

Esquema da Representação Intermediária:
{{{{IR_SCHEMA_JSON}}}}

Exemplos few-shot:
{{{{EXEMPLOS_FEW_SHOT_R1}}}}

Entrada de teste em linguagem natural:
{{{{NL_TESTE}}}}

Regras específicas:
1. Responda apenas com um objeto JSON válido da Representação Intermediária.
2. Use aspas duplas em todas as chaves e strings.
3. Não use vírgula após o último item de objetos ou listas.
4. Não gere a expressão Nile nesta etapa.
5. Use as demonstrações enriquecidas com R1 somente como padrão estrutural.
6. Preserve a intenção da entrada em linguagem natural.
7. Não gere R1 para o exemplo de teste.
8. Preserve em inglês termos, nomes, valores e entidades extraídos da entrada de teste.

Representação Intermediária:
""",
)


# ----------------------------------------------------------
# 9.16 TP10 - Estratégia E: geração de Nile com R2
# ----------------------------------------------------------

registrar_prompt(
    tp="TP10",
    prompt_id="f3e_gerar_nile_com_r2",
    arquivo="f3e_gerar_nile_com_r2.txt",
    fase="F3-E",
    modelo="Qwen2.5-1.5B-Instruct",
    estrategia="E",
    nome_tabela="Geração de Nile com R2",
    objetivo="Gerar Nile a partir da IR usando demonstrações enriquecidas com R2.",
    placeholders=["EXEMPLOS_FEW_SHOT_R2", "IR_TESTE"],
    texto=f"""
{abertura_modelo_aluno}

{regras_saida_modelo_aluno}

Tarefa:
Converta a Representação Intermediária abaixo em uma expressão completa em Nile utilizando os exemplos demonstrativos enriquecidos com R2.

Exemplos few-shot:
{{{{EXEMPLOS_FEW_SHOT_R2}}}}

Representação Intermediária do teste:
{{{{IR_TESTE}}}}

Regras específicas:
1. Responda apenas com a expressão final em Nile.
2. A saída deve ser completa, válida e analisável pela gramática Nile.
3. Use somente as informações da Representação Intermediária do teste e o padrão composicional das demonstrações enriquecidas com R2.
4. Não repita a Representação Intermediária.
5. Não gere R2 para o exemplo de teste.

Saída Nile:
""",
)


# ----------------------------------------------------------
# 9.17 TP11 - Autocorreção da saída Nile
# ----------------------------------------------------------

registrar_prompt(
    tp="TP11",
    prompt_id="f3_autocorrecao_nile",
    arquivo="f3_autocorrecao_nile.txt",
    fase="F3-B / F3-C / F3-D / F3-E",
    modelo="Qwen2.5-1.5B-Instruct",
    estrategia="B_C_D_E",
    nome_tabela="Autocorreção da saída Nile",
    objetivo="Corrigir uma saída Nile rejeitada com base no feedback do validador.",
    placeholders=[
        "NL_TESTE",
        "ARTEFATOS_AUXILIARES",
        "NILE_REJEITADA",
        "FEEDBACK_VALIDACAO_NILE",
        "EXEMPLOS_FEW_SHOT",
    ],
    texto=f"""
{abertura_modelo_aluno}

{regras_saida_modelo_aluno}

Tarefa:
Revise a saída Nile rejeitada pelo validador e produza uma versão corrigida.

Entrada de teste em linguagem natural:
{{{{NL_TESTE}}}}

Artefatos auxiliares disponíveis:
{{{{ARTEFATOS_AUXILIARES}}}}

Saída Nile rejeitada:
{{{{NILE_REJEITADA}}}}

Feedback do validador:
{{{{FEEDBACK_VALIDACAO_NILE}}}}

Exemplos few-shot:
{{{{EXEMPLOS_FEW_SHOT}}}}

Regras específicas:
1. Responda apenas com a expressão final corrigida em Nile.
2. A saída deve ser completa, válida e analisável pela gramática Nile.
3. Corrija os problemas apontados pelo feedback do validador.
4. Preserve a intenção da entrada em linguagem natural.
5. Use os artefatos auxiliares somente quando eles forem fornecidos.
6. Modifique apenas o que for necessário para corrigir a saída.
7. Não gere justificativa.
8. Não gere Representação Intermediária.

Saída Nile corrigida:
""",
)


# ----------------------------------------------------------
# 9.18 Ordem oficial dos Templates de Prompt
# ----------------------------------------------------------

ordem_prompts = [
    "f2_geracao_r1",
    "f2_geracao_r2",
    "f2_geracao_r3",
    "f2_revisao_justificativa_reprovada",
    "f3a_traducao_direta",
    "f3c_gerar_ir_teste",
    "f3c_gerar_nile_a_partir_ir",
    "f3d_traducao_com_r3",
    "f3e_gerar_ir_com_r1",
    "f3e_gerar_nile_com_r2",
    "f3_autocorrecao_nile",
]


# ----------------------------------------------------------
# 9.19 Verificação da numeração e dos identificadores
# ----------------------------------------------------------

if len(prompts) != EXPECTED_TEMPLATES:
    raise AssertionError(
        f"Quantidade de templates inválida: {len(prompts)}. "
        f"Esperado: {EXPECTED_TEMPLATES}."
    )

if set(prompts) != set(ordem_prompts):
    raise AssertionError("A ordem oficial não corresponde aos templates registrados.")

tp_esperados = [f"TP{i}" for i in range(1, EXPECTED_TEMPLATES + 1)]
tp_obtidos = [prompts[prompt_id]["tp"] for prompt_id in ordem_prompts]

if tp_obtidos != tp_esperados:
    raise AssertionError(
        f"Numeração dos templates inválida: {tp_obtidos}. "
        f"Esperado: {tp_esperados}."
    )

if len({item["prompt_id"] for item in prompts.values()}) != EXPECTED_TEMPLATES:
    raise AssertionError("Existem prompt_id duplicados.")


# ----------------------------------------------------------
# 9.20 Verificação dos placeholders
# ----------------------------------------------------------

for prompt_id in ordem_prompts:
    item = prompts[prompt_id]
    for placeholder in item["placeholders"]:
        marcador = "{{" + placeholder + "}}"
        if marcador not in item["texto"]:
            raise AssertionError(
                f"O placeholder {marcador} não aparece no template {item['tp']}."
            )


# ----------------------------------------------------------
# 9.21 Salvamento individual dos prompts
# ----------------------------------------------------------

PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

for prompt_id in ordem_prompts:
    item = prompts[prompt_id]
    caminho_prompt = PROMPTS_DIR / item["arquivo"]
    caminho_prompt.write_text(
        item["texto"].strip() + "\n",
        encoding="utf-8",
    )
    item["arquivo_relativo"] = str(caminho_prompt.relative_to(F0_DIR))


# ----------------------------------------------------------
# 9.22 Criação do índice da biblioteca de prompts
# ----------------------------------------------------------

biblioteca_prompts = {
    "dataset": DATASET_ID,
    "total_prompts": len(prompts),
    "ordem_prompts": ordem_prompts,
    "diretorio_prompts": PROMPTS_DIR.name,
    "prompts": {
        prompt_id: {
            chave: valor
            for chave, valor in prompts[prompt_id].items()
            if chave != "texto"
        }
        for prompt_id in ordem_prompts
    },
}

salvar_json(biblioteca_prompts, PROMPT_LIBRARY_PATH)

templates = [prompts[prompt_id] for prompt_id in ordem_prompts]


# ----------------------------------------------------------
# 9.23 Tabela-resumo dos prompts
# ----------------------------------------------------------

df_templates = pd.DataFrame([
    {
        "tp": prompts[prompt_id]["tp"],
        "prompt_id": prompts[prompt_id]["prompt_id"],
        "fase": prompts[prompt_id]["fase"],
        "modelo": prompts[prompt_id]["modelo"],
        "estrategia": prompts[prompt_id]["estrategia"],
        "nome": prompts[prompt_id]["nome_tabela"],
        "arquivo": prompts[prompt_id]["arquivo_relativo"],
        "objetivo": prompts[prompt_id]["objetivo"],
        "placeholders": ", ".join(prompts[prompt_id]["placeholders"]),
    }
    for prompt_id in ordem_prompts
])

exibir_tabela(
    df_templates,
    "Biblioteca oficial dos 11 prompts",
    altura_px=560,
)


# ----------------------------------------------------------
# 9.24 Saída do bloco
# ----------------------------------------------------------

print("Bloco 9 concluído")
print(f"Templates esperados: {EXPECTED_TEMPLATES}")
print(f"Templates registrados: {len(prompts)}")
print("Status: OK")

TP,ID do prompt,fase,modelo,estratégia,nome,arquivo,objetivo,placeholders
TP1,f2_geracao_r1,F2,DeepSeek-V4,preparacao_r1,Geração da justificativa explícita R1,prompts/f2_geracao_r1.txt,Gerar R1 para explicar a passagem da linguagem natural para a Representação Intermediária.,IR_JSON
TP2,f2_geracao_r2,F2,DeepSeek-V4,preparacao_r2,Geração da justificativa explícita R2,prompts/f2_geracao_r2.txt,Gerar R2 para explicar a passagem da Representação Intermediária para Nile.,IR_JSON
TP3,f2_geracao_r3,F2,DeepSeek-V4,preparacao_r3,Geração da justificativa explícita R3,prompts/f2_geracao_r3.txt,Gerar R3 para explicar a passagem direta da linguagem natural para Nile.,CAMPI_JSON
TP4,f2_revisao_justificativa_reprovada,F2,DeepSeek-V4,revisao_r1_r2_r3,"Revisão de justificativa R1, R2 ou R3",prompts/f2_revisao_justificativa_reprovada.txt,Corrigir uma justificativa reprovada pela validação da F2.,"TIPO_JUSTIFICATIVA, EXEMPLO_JSON, JUSTIFICATIVA_REPROVADA, ERROS_VALIDACAO_JSON"
TP5,f3a_traducao_direta,F3-A,Qwen2.5-1.5B-Instruct,A,Tradução direta para Nile,prompts/f3a_traducao_direta.txt,Traduzir diretamente uma entrada em linguagem natural para Nile.,"EXEMPLOS_FEW_SHOT, NL_TESTE"
TP6,f3c_gerar_ir_teste,F3-C,Qwen2.5-1.5B-Instruct,C,Geração da Representação Intermediária do teste,prompts/f3c_gerar_ir_teste.txt,Gerar uma IR para a entrada de teste sem usar a expressão Nile de referência.,"IR_SCHEMA_JSON, EXEMPLOS_FEW_SHOT_IR, NL_TESTE"
TP7,f3c_gerar_nile_a_partir_ir,F3-C,Qwen2.5-1.5B-Instruct,C,Geração de Nile a partir da Representação Intermediária,prompts/f3c_gerar_nile_a_partir_ir.txt,Gerar Nile a partir da IR produzida para o exemplo de teste.,"EXEMPLOS_FEW_SHOT_IR_NILE, IR_TESTE"
TP8,f3d_traducao_com_r3,F3-D,Qwen2.5-1.5B-Instruct,D,Tradução direta com demonstrações enriquecidas com R3,prompts/f3d_traducao_com_r3.txt,Traduzir linguagem natural para Nile usando demonstrações com R3.,"EXEMPLOS_FEW_SHOT_R3, NL_TESTE"
TP9,f3e_gerar_ir_com_r1,F3-E,Qwen2.5-1.5B-Instruct,E,Geração da Representação Intermediária com R1,prompts/f3e_gerar_ir_com_r1.txt,Gerar uma IR usando demonstrações enriquecidas com R1.,"IR_SCHEMA_JSON, EXEMPLOS_FEW_SHOT_R1, NL_TESTE"
TP10,f3e_gerar_nile_com_r2,F3-E,Qwen2.5-1.5B-Instruct,E,Geração de Nile com R2,prompts/f3e_gerar_nile_com_r2.txt,Gerar Nile a partir da IR usando demonstrações enriquecidas com R2.,"EXEMPLOS_FEW_SHOT_R2, IR_TESTE"


Bloco 9 concluído
Templates esperados: 11
Templates registrados: 11
Status: OK


## Bloco 10 - Manifesto e empacotamento final

### Objetivo

Este bloco encerra a F0, verifica os artefatos obrigatórios, registra a proveniência da execução e cria o pacote operacional consumido pelas demais fases.

### Auditoria final

Antes do empacotamento, o bloco confirma a existência e a legibilidade de:

- fonte original preservada;
- CAMPI canônico e changelog;
- folds;
- gramática;
- núcleo formal;
- métricas;
- testes do validador;
- validação das referências;
- biblioteca e arquivos individuais de prompts.

Arquivos temporários, caches e bytecodes são removidos.

### Manifesto

`manifest.json` registra:

- identificação da fase e do conjunto CAMPI;
- data e ambiente de execução;
- versões das bibliotecas;
- semente e parâmetros fixos;
- quantidade de exemplos, folds e templates;
- resultados das validações;
- inventário dos arquivos;
- tamanho e SHA-256 de cada artefato.

O inventário é construído sem autorreferência: o manifesto não tenta registrar o próprio hash antes de ser finalizado.

### Pacote final

O arquivo produzido é:

```text
f0_operacional.zip
```

Ele contém os artefatos congelados da F0, incluindo a pasta `prompts/`. A estrutura do ZIP é verificada depois da criação para garantir que não existam arquivos faltantes, adicionais ou duplicados.

### Limites registrados

O pacote representa validação operacional sobre as 50 referências concretas do CAMPI e sobre classes negativas dirigidas. Ele não declara prova matemática de correção da linguagem, implantação de políticas ou execução em rede.

### Resultado esperado

A tabela final resume quantidades, validações e localização do ZIP. A F0 só é considerada concluída quando todos os arquivos previstos estão presentes e o pacote pode ser aberto e auditado.

In [10]:
# ----------------------------------------------------------
# 10.1 Verificação dos artefatos obrigatórios
# ----------------------------------------------------------

artefatos_obrigatorios = [
    CAMPI_ORIGINAL_PATH,
    CAMPI_CANONICAL_PATH,
    CAMPI_CHANGELOG_PATH,
    FOLDS_PATH,
    GRAMMAR_PATH,
    NILE_CORE_PATH,
    NILE_METRICS_PATH,
    VALIDATOR_TESTS_PATH,
    VALIDATION_REFERENCES_PATH,
    PROMPT_LIBRARY_PATH,
]

artefatos_obrigatorios.extend(
    PROMPTS_DIR / prompts[prompt_id]["arquivo"]
    for prompt_id in ordem_prompts
)

faltantes = [str(caminho) for caminho in artefatos_obrigatorios if not caminho.exists()]
if faltantes:
    raise FileNotFoundError(
        "Artefatos obrigatórios ausentes:\n" + "\n".join(faltantes)
    )


# ----------------------------------------------------------
# 10.2 Registro do ambiente de execução
# ----------------------------------------------------------

ambiente = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "lark": lark.__version__,
    "scikit_learn": versao_pacote("scikit-learn"),
}


# ----------------------------------------------------------
# 10.3 Remoção de arquivos temporários de execução
# ----------------------------------------------------------

for diretorio_cache in F0_DIR.rglob("__pycache__"):
    if diretorio_cache.is_dir():
        shutil.rmtree(diretorio_cache)

for arquivo_temporario in F0_DIR.rglob("*.pyc"):
    if arquivo_temporario.is_file():
        arquivo_temporario.unlink()


# ----------------------------------------------------------
# 10.4 Construção do inventário sem autorreferência
# ----------------------------------------------------------

def inventariar_arquivos_sem_manifesto():
    registros = []
    for caminho in sorted(F0_DIR.rglob("*")):
        if not caminho.is_file() or caminho.resolve() == MANIFEST_PATH.resolve():
            continue
        registros.append({
            "arquivo": str(caminho.relative_to(F0_DIR)),
            "tamanho_bytes": caminho.stat().st_size,
            "sha256": calcular_sha256(caminho),
        })
    return registros


inventario = inventariar_arquivos_sem_manifesto()


# ----------------------------------------------------------
# 10.5 Criação do manifesto
# ----------------------------------------------------------

manifesto = {
    "fase": FASE,
    "descricao": "Preparação da base formal e experimental CAMPI.",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": {
        "id": DATASET_ID,
        "source_file": CAMPI_ORIGINAL_PATH.name,
        "source_sha256": calcular_sha256(CAMPI_ORIGINAL_PATH),
        "total_examples": len(df_campi),
        "canonical_file": CAMPI_CANONICAL_PATH.name,
        "changelog_file": CAMPI_CHANGELOG_PATH.name,
        "changed_examples": len(df_changelog),
    },
    "validation": {
        "grammar_file": GRAMMAR_PATH.name,
        "core_module": NILE_CORE_PATH.name,
        "test_file": VALIDATOR_TESTS_PATH.name,
        "reference_validation_file": VALIDATION_REFERENCES_PATH.name,
        "positive_tests": len(casos_positivos),
        "negative_tests": len(casos_negativos),
        "psr_reference": float(psr_referencia),
        "structural_reference_rate": float(taxa_estrutural_referencia),
        "roundtrip_passed": bool(df_validacao_referencias["roundtrip_ok"].all()),
    },
    "metrics": {
        "module": NILE_METRICS_PATH.name,
        "names": ["PSR", "EM", "ED", "NED", "SLA-S", "SLA-F"],
        "embeddings_used": False,
    },
    "folds": {
        "file": FOLDS_PATH.name,
        "seed": SEED,
        "n_folds": N_FOLDS,
        "test_per_fold": EXPECTED_TEST_PER_FOLD,
        "train_per_fold": EXPECTED_EXAMPLES - EXPECTED_TEST_PER_FOLD,
        "every_example_tested_once": bool((ocorrencias_teste == 1).all()),
    },
    "prompts": {
        "library_file": PROMPT_LIBRARY_PATH.name,
        "directory": PROMPTS_DIR.name,
        "total": len(templates),
        "files": [prompts[prompt_id]["arquivo_relativo"] for prompt_id in ordem_prompts],
    },
    "environment": ambiente,
    "files": inventario,
}

salvar_json(manifesto, MANIFEST_PATH)


# ----------------------------------------------------------
# 10.6 Criação do ZIP operacional
# ----------------------------------------------------------

if ZIP_F0_PATH.exists():
    ZIP_F0_PATH.unlink()

with zipfile.ZipFile(ZIP_F0_PATH, "w", zipfile.ZIP_DEFLATED) as arquivo_zip:
    for caminho in sorted(F0_DIR.rglob("*")):
        if caminho.is_file():
            arquivo_zip.write(
                caminho,
                arcname=str(caminho.relative_to(F0_DIR)),
            )


# ----------------------------------------------------------
# 10.7 Verificação do conteúdo do ZIP
# ----------------------------------------------------------

with zipfile.ZipFile(ZIP_F0_PATH, "r") as arquivo_zip:
    arquivos_zip = sorted(arquivo_zip.namelist())

arquivos_esperados_zip = sorted([
    "extraction_campi.json",
    "campi_canonical.csv",
    "campi_changelog.csv",
    "folds.csv",
    "nile_subset.lark",
    "nile_core.py",
    "nile_metrics.py",
    "validator_tests.jsonl",
    "validation_references.csv",
    "biblioteca_prompts.json",
    "manifest.json",
] + [
    prompts[prompt_id]["arquivo_relativo"]
    for prompt_id in ordem_prompts
])

if arquivos_zip != arquivos_esperados_zip:
    raise AssertionError(
        "O conteúdo do ZIP difere do conjunto esperado.\n"
        f"Esperado: {arquivos_esperados_zip}\n"
        f"Obtido: {arquivos_zip}"
    )


# ----------------------------------------------------------
# 10.8 Tabela de resumo final
# ----------------------------------------------------------

resumo_final = pd.DataFrame([{
    "fase": FASE,
    "dataset": DATASET_ID,
    "exemplos": len(df_campi),
    "referencias_sintaticamente_validas": int(df_validacao_referencias["syntax_valid"].sum()),
    "referencias_estruturalmente_validas": int(df_validacao_referencias["structural_valid"].sum()),
    "psr_referencia": round(float(psr_referencia), 4),
    "folds": N_FOLDS,
    "testes_por_fold": EXPECTED_TEST_PER_FOLD,
    "prompts": len(templates),
    "arquivos_no_zip": len(arquivos_zip),
    "zip": str(ZIP_F0_PATH),
    "status": "OK",
}])

exibir_tabela(
    resumo_final,
    "Resumo final da F0",
    altura_px=240,
)


# ----------------------------------------------------------
# 10.9 Saída final do notebook
# ----------------------------------------------------------

print("Bloco 10 concluído")
print(f"ZIP gerado: {ZIP_F0_PATH}")
print(f"Tamanho do ZIP: {ZIP_F0_PATH.stat().st_size:,} bytes")
print("F0 concluída com sucesso")
print("Status: OK")

fase,conjunto de dados,exemplos,referências sintaticamente válidas,referências estruturalmente válidas,PSR da referência,folds,testes por fold,prompts,arquivos no ZIP,ZIP,status
F0,CAMPI,50,50,50,1.0,5,10,11,22,/kaggle/working/f0_operacional.zip,OK


Bloco 10 concluído
ZIP gerado: /kaggle/working/f0_operacional.zip
Tamanho do ZIP: 37,262 bytes
F0 concluída com sucesso
Status: OK
